# OOF Stacking

RF, TextCNN, BERTweet -> OOF predictions -> Logistic Regression -> validation threshold -> test evaluation.


## 1. Configuration

Experiment paths, seeds, and execution options.


In [1]:
from pathlib import Path
import gc
import hashlib
import json
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import sklearn
import joblib
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
from IPython.display import display

BASE = Path('/content/drive_research/MyDrive/thesis/saved_results')
MULTISEED = BASE / '17_conference_revision/multiseed'
STACKING = BASE / '17_conference_revision/stacking_oof'
SEEDS = [42, 43, 44, 45, 46]
ENABLE_TRAINING = False
ENABLE_META_FIT = False
ENABLE_THRESHOLD_SEARCH = False
TRAINING_STAGE = None

METRICS = ['accuracy', 'precision_ASD', 'recall_ASD', 'f1_ASD',
           'macro_f1', 'recall_non_ASD', 'f1_non_ASD', 'roc_auc']

def require_files(paths):
    missing = [str(Path(p)) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError('Missing files:\n' + '\n'.join(missing))

def read_json(path):
    require_files([path])
    return json.loads(Path(path).read_text())

def require_training(stage, meta=False):
    if ENABLE_TRAINING is not True or TRAINING_STAGE != stage:
        raise RuntimeError('Training is disabled.')
    if meta and not (ENABLE_META_FIT is True and ENABLE_THRESHOLD_SEARCH is True):
        raise RuntimeError('Meta fitting and threshold search are disabled.')
    if meta:
        seeds = [42] if stage == 'meta_42' else [43,44,45,46]
        targets = [STACKING/f'seed_{s}'/f for s in seeds
                   for f in ['PRIVATE_meta_model.joblib','stacking_selected_settings.json','test_evaluation_record.json']]
        if any(p.exists() for p in targets):
            raise FileExistsError('Meta checkpoint or settings already exist.')


    if stage.startswith('folds_'):
        seeds = [42] if stage=='folds_42' else [43,44,45,46]
        targets = [STACKING/f'seed_{s}'/name for s in seeds
                   for name in ['PRIVATE_training_user_fold_plan.csv','fold_summary.csv','experiment_plan.json']]
    elif stage.startswith(('rf_','textcnn_','bertweet_')):
        family,group = stage.split('_',1)
        name = {'rf':'RF','textcnn':'TextCNN','bertweet':'BERTweet'}[family]
        seeds = [42] if group=='42' else [43,44,45,46]
        targets = [p for s in seeds for p in (STACKING/f'seed_{s}').glob(f'*{name}*')]
    elif stage=='transformer':
        targets = [p for s in [42,43,44,45,46] for p in (MULTISEED/f'seed_{s}').glob('*TFM_*')]
        targets += [MULTISEED/name for name in ['Transformer_multiseed_results.csv','Transformer_multiseed_mean_SD.csv',
                                              'All_models_multiseed_results.csv','All_models_multiseed_mean_SD.csv']]
    else:
        targets = []
    if any(p.exists() for p in targets):
        raise FileExistsError('Stage outputs already exist.')

def partition(data, seed):
    folder = MULTISEED / f'seed_{seed}'
    require_files([folder/f'{part}_users.csv' for part in ['train','val','test']])
    parts = {}
    for part in ['train','val','test']:
        names = pd.read_csv(folder/f'{part}_users.csv')['username']
        assert names.notna().all() and names.is_unique
        parts[part] = set(names)
    assert tuple(len(parts[p]) for p in ['train','val','test']) == (146,37,61)
    assert not (parts['train'] & parts['val'] or parts['train'] & parts['test'] or parts['val'] & parts['test'])
    assert set().union(*parts.values()) == set(data['username'])
    positions = {p:np.flatnonzero(data['username'].isin(names).to_numpy()) for p,names in parts.items()}
    labels = data.iloc[positions['test']].groupby('username',sort=True)['label'].first()
    assert labels.value_counts().to_dict() == {1:54,0:7}
    return parts, positions

def check_users(users, data, positions, prediction='prediction', probability='ASD_probability', vote='vote_fraction'):
    expected = data.iloc[positions].groupby('username',sort=True)['label'].first()
    assert users.index.is_unique and set(users.index) == set(expected.index)
    users = users.loc[expected.index].copy()
    assert np.array_equal(users['label'].to_numpy(), expected.to_numpy())
    assert users[prediction].isin([0,1]).all()
    assert np.isfinite(users[probability]).all() and users[probability].between(0,1).all()
    assert users[vote].between(0,1).all()
    assert np.array_equal(users[prediction].to_numpy(), (users[vote]>=0.5).astype(int).to_numpy())
    return users

def check_metric_row(row, values):
    for key,value in values.items():
        assert key in row.index and np.isclose(float(row[key]),value,atol=1e-8,rtol=0), key


In [2]:
MODEL_ORDER = ['RF','TextCNN','BERTweet']
EXPECTED_THRESHOLDS = {42:0.49,43:0.46,44:0.46,45:0.50,46:0.45}
N_BOOTSTRAP = 5000
RECOMPUTE_CI = False
# Reuse signed experiment probabilities; checkpoint comparison is opt-in.
PREDICT_FROM_CHECKPOINT = False

def read_probability(path, signature, positions, prefix=None):
    require_files([path])
    with np.load(path, allow_pickle=False) as saved:
        assert saved['signature'].item() == signature, str(path)
        row_key = 'row_positions' if prefix is None else f'{prefix}_row_positions'
        probability_key = 'probabilities' if prefix is None else f'{prefix}_probabilities'
        assert np.array_equal(saved[row_key],positions), str(path)
        values = saved[probability_key].copy()
    assert values.shape == (len(positions),)
    assert np.isfinite(values).all() and ((values>=0)&(values<=1)).all()
    return values

def stacking_context(data, seed):
    out = STACKING/f'seed_{seed}'
    parts, positions = partitions[seed]
    require_files([out/'PRIVATE_training_user_fold_plan.csv'])
    plan = pd.read_csv(out/'PRIVATE_training_user_fold_plan.csv')
    assert plan['username'].is_unique and set(plan['username']) == parts['train']
    assert set(plan['heldout_fold']) == {1,2,3}
    user_labels = data.groupby('username',sort=True)['label'].first()
    assert np.array_equal(plan['label'].to_numpy(),user_labels.loc[plan['username']].to_numpy())
    source_bytes = pd.util.hash_pandas_object(data,index=True).values.tobytes()+plan.to_csv(index=False).encode()
    configs = {name:read_json(out/f'{name}_parameters.json') for name in MODEL_ORDER}
    if seed == 42:
        rf_signature = hashlib.sha256(source_bytes+configs['RF']['sklearn_version'].encode()
            +b'RF100_TFIDF10000_ngram12_minDF2_sublinear_seed42_v1').hexdigest()
        cnn_signature = hashlib.sha256(source_bytes+json.dumps(configs['TextCNN']['parameters'],sort_keys=True).encode()).hexdigest()
        bert_signature = hashlib.sha256(source_bytes+json.dumps(configs['BERTweet']['parameters'],sort_keys=True).encode()).hexdigest()
    else:
        source_bytes += json.dumps({p:sorted(v) for p,v in parts.items()},sort_keys=True).encode()
        rf_signature = hashlib.sha256(source_bytes+json.dumps(configs['RF']['parameters'],sort_keys=True).encode()).hexdigest()
        cnn_signature = hashlib.sha256(source_bytes+rf_signature.encode()
            +json.dumps(configs['TextCNN']['parameters'],sort_keys=True).encode()
            +json.dumps(configs['TextCNN']['tokenizer_settings'],sort_keys=True).encode()).hexdigest()
        unsigned = {k:v for k,v in configs['BERTweet'].items() if k!='signature'}
        bert_signature = hashlib.sha256(source_bytes+json.dumps(unsigned,sort_keys=True).encode()).hexdigest()
    for name,expected in zip(MODEL_ORDER,[rf_signature,cnn_signature,bert_signature]):
        assert configs[name]['signature'] == expected, (seed,name,'signature')
    for name in ['TextCNN','BERTweet']:
        assert configs[name]['source_RF_signature'] == rf_signature
    return out, positions, plan, configs, source_bytes

def frozen_settings(seed):
    out = STACKING/f'seed_{seed}'
    path = out/'stacking_selected_settings.json'
    settings = read_json(path)
    assert settings['seed']==seed and settings['model_order']==MODEL_ORDER
    assert settings['threshold_selection_metric']=='user-level Macro-F1'
    assert np.isclose(settings['selected_threshold'],EXPECTED_THRESHOLDS[seed],atol=1e-12,rtol=0)
    record_path = out/'test_evaluation_record.json'
    if record_path.is_file():
        record = read_json(record_path)
        key = 'selection_file_sha256' if seed==42 else 'selected_settings_sha256'
        assert record[key] == hashlib.sha256(path.read_bytes()).hexdigest()
        if seed==42:
            assert record['threshold_retuned_on_test'] is False
        else:
            assert record['test_used_for_retuning'] is False
            assert record['n_bootstrap']==5000 and record['bootstrap_seed']==seed
    return settings


## 2. Dataset and Folds

146/37/61 users in training/validation/test; three folds within training. Dataset row order is preserved.


In [3]:
if not (BASE/'02_preprocessing/cleaned_data.pkl').is_file():
    try:
        from google.colab import drive
    except ImportError:
        raise FileNotFoundError(f'Dataset path: {BASE}') from None
    drive.mount('/content/drive_research')


Mounted at /content/drive_research


In [4]:
require_files([BASE/'02_preprocessing/cleaned_data.pkl'])
data = pd.read_pickle(BASE/'02_preprocessing/cleaned_data.pkl')[
    ['username','label','cleaned_tweet']
].reset_index(drop=True)
assert data.notna().all().all()
assert data['label'].isin([0,1]).all()
assert data.groupby('username')['label'].nunique().eq(1).all()
assert data['username'].nunique() == 244
partitions = {seed:partition(data,seed) for seed in SEEDS}
display(pd.DataFrame([{'tweets':len(data),'users':data['username'].nunique()}]))


,tweets,users
0,920070,244


In [ ]:
REVISION = BASE / '17_conference_revision'
LINGUISTIC = REVISION / 'linguistic_robustness'
REPORT_BASE_MODELS = ['Logistic Regression', 'SVM', 'Random Forest', 'Bi-LSTM',
                      'TextCNN', 'DistilBERT', 'RoBERTa', 'BERTweet']

def read_report_csv(path, **kwargs):
    require_files([path])
    return pd.read_csv(path, **kwargs)

def show_result(title, function):
    print(title)
    try:
        table = function()
    except FileNotFoundError as error:
        print(str(error))
        return None
    display(table)
    return table


def saved_fold_overview():
    tables = []
    for seed in SEEDS:
        table = read_report_csv(STACKING/f'seed_{seed}/fold_summary.csv')
        table.insert(0,'seed',seed)
        tables.append(table)
    return pd.concat(tables,ignore_index=True)

def saved_meta_overview():
    rows,coefficient_rows = [],[]
    for seed in SEEDS:
        settings = frozen_settings(seed)
        rows.append({'seed':seed,'threshold':settings['selected_threshold'],
                     'validation_user_macro_f1':settings['validation_user_macro_f1'],
                     'intercept':settings['meta_intercept']})
        table = read_report_csv(STACKING/f'seed_{seed}/meta_learner_coefficients.csv')
        assert table['base_model'].tolist()==settings['model_order']
        table.insert(0,'seed',seed)
        coefficient_rows.append(table)
    display(pd.DataFrame(rows).round(4))
    return pd.concat(coefficient_rows,ignore_index=True).pivot(
        index='seed',columns='base_model',values='coefficient'
    ).reindex(columns=MODEL_ORDER).round(4)

def saved_probability_diagnostics():
    tables = []
    for seed in SEEDS:
        table = read_report_csv(STACKING/f'seed_{seed}/base_probability_diagnostics.csv')
        table.insert(0,'seed',seed)
        tables.append(table)
    return pd.concat(tables,ignore_index=True).round(4)


In [6]:
def prepare_folds_seed42():
    require_training('folds_42', meta=False)
    import json
    from pathlib import Path

    import numpy as np
    import pandas as pd
    from sklearn.model_selection import StratifiedKFold

    BASE = Path("/content/drive_research/MyDrive/thesis/saved_results")
    SPLIT_DIR = BASE / "17_conference_revision/multiseed/seed_42"


    OUT = BASE / "17_conference_revision/stacking_oof/seed_42"
    OUT.mkdir(parents=True, exist_ok=True)

    data = pd.read_pickle(
        BASE / "02_preprocessing/cleaned_data.pkl"
    ).reset_index(drop=True)

    assert data[["username", "label", "cleaned_tweet"]].notna().all().all()
    assert data.groupby("username")["label"].nunique().eq(1).all()

    users = (
        data.groupby("username", sort=True)
        .agg(label=("label", "first"), n_tweets=("label", "size"))
    )

    def read_user_list(filename):
        table = pd.read_csv(SPLIT_DIR / filename)
        assert "username" in table.columns, table.columns.tolist()
        assert table["username"].notna().all()
        assert table["username"].is_unique
        return table["username"].tolist()

    train_users = read_user_list("train_users.csv")
    val_users = read_user_list("val_users.csv")
    test_users = read_user_list("test_users.csv")

    train_set, val_set, test_set = map(
        set, [train_users, val_users, test_users]
    )

    assert not (train_set & val_set)
    assert not (train_set & test_set)
    assert not (val_set & test_set)
    assert train_set | val_set | test_set == set(users.index)
    assert (len(train_users), len(val_users), len(test_users)) == (146, 37, 61)

    training_users = (
        users.loc[train_users]
        .reset_index()
        .sort_values("username")
        .reset_index(drop=True)
    )

    assert training_users["label"].value_counts().min() >= 3

    fold_assignment = np.full(len(training_users), -1, dtype=int)
    splitter = StratifiedKFold(
        n_splits=3, shuffle=True, random_state=42
    )

    fold_summary = []

    for fold, (fit_idx, heldout_idx) in enumerate(
        splitter.split(training_users["username"], training_users["label"]),
        start=1
    ):
        fit_users = training_users.iloc[fit_idx]
        heldout_users = training_users.iloc[heldout_idx]

        fit_set = set(fit_users["username"])
        heldout_set = set(heldout_users["username"])

        assert not (fit_set & heldout_set)
        assert not ((fit_set | heldout_set) & (val_set | test_set))

        fold_assignment[heldout_idx] = fold


        fit_tweets_by_label = (
            fit_users.groupby("label")["n_tweets"].sum()
            .reindex([0, 1], fill_value=0)
        )

        fold_summary.append({
            "fold": fold,
            "fit_users": len(fit_users),
            "fit_nonASD": int((fit_users["label"] == 0).sum()),
            "fit_ASD": int((fit_users["label"] == 1).sum()),
            "heldout_users": len(heldout_users),
            "heldout_nonASD": int((heldout_users["label"] == 0).sum()),
            "heldout_ASD": int((heldout_users["label"] == 1).sum()),
            "fit_tweets_before_RUS": int(fit_tweets_by_label.sum()),
            "fit_tweets_after_RUS": int(2 * fit_tweets_by_label.min())
        })

    assert (fold_assignment >= 1).all()
    training_users["heldout_fold"] = fold_assignment


    training_users.to_csv(
        OUT / "PRIVATE_training_user_fold_plan.csv", index=False
    )

    summary = pd.DataFrame(fold_summary)
    summary.to_csv(OUT / "fold_summary.csv", index=False)

    with open(OUT / "experiment_plan.json", "w") as f:
        json.dump({
            "status": "fold plan only; models not trained",
            "seed": 42,
            "n_folds": 3,
            "base_models": ["Random Forest", "TextCNN", "BERTweet"],
            "meta_model": "Logistic Regression",
            "meta_training": "training-tweet user-disjoint OOF probabilities",
            "validation_role": "threshold selection using user-level Macro-F1",
            "threshold_grid": {
                "start": 0.05, "stop": 0.95, "step": 0.01
            },
            "threshold_tie_rule": "closest to 0.50, then smaller threshold",
            "user_prediction": "majority vote; ties assigned ASD",
            "user_ROC_AUC": "mean tweet ASD probability",
            "test_role": "evaluation after settings are fixed",
            "base_model_selection_note":
                "Base models retained from the previous exploratory experiment; "
                "not selected anew using the held-out test."
        }, f, indent=2)

    print('Split: train=146 | validation=37 | test=61')
    print('Folds')
    print(summary.to_string(index=False))


In [7]:
def prepare_folds_seed43_46():
    require_training('folds_43_46', meta=False)
    import json
    from pathlib import Path

    import numpy as np
    import pandas as pd
    from sklearn.model_selection import StratifiedKFold

    BASE = Path("/content/drive_research/MyDrive/thesis/saved_results")
    ROOT = BASE / "17_conference_revision"

    data = pd.read_pickle(
        BASE / "02_preprocessing/cleaned_data.pkl"
    )[["username", "label", "cleaned_tweet"]].reset_index(drop=True)

    assert data.notna().all().all()
    assert data["label"].isin([0, 1]).all()
    assert data.groupby("username")["label"].nunique().eq(1).all()

    users = (
        data.groupby("username", sort=True)
        .agg(label=("label", "first"), n_tweets=("label", "size"))
    )
    assert len(users) == 244


    plans = []

    for seed in range(43, 47):
        split_dir = ROOT / f"multiseed/seed_{seed}"
        out = ROOT / f"stacking_oof/seed_{seed}"

        parts = {}
        for part in ["train", "val", "test"]:
            table = pd.read_csv(split_dir / f"{part}_users.csv")
            assert table["username"].notna().all()
            assert table["username"].is_unique
            parts[part] = set(table["username"])

        train_set, val_set, test_set = [
            parts[p] for p in ["train", "val", "test"]
        ]

        assert not (train_set & val_set)
        assert not (train_set & test_set)
        assert not (val_set & test_set)
        assert train_set | val_set | test_set == set(users.index)
        assert tuple(map(len, [train_set, val_set, test_set])) == (146, 37, 61)
        assert users.loc[sorted(test_set), "label"].value_counts().to_dict() == {
            1: 54, 0: 7
        }

        training_users = (
            users.loc[sorted(train_set)]
            .reset_index()
            .sort_values("username")
            .reset_index(drop=True)
        )

        assert training_users["label"].value_counts().min() >= 3

        assignment = np.full(len(training_users), -1, dtype=int)
        splitter = StratifiedKFold(
            n_splits=3, shuffle=True, random_state=seed
        )
        rows = []

        for fold, (fit_idx, heldout_idx) in enumerate(
            splitter.split(training_users["username"], training_users["label"]),
            start=1
        ):
            fit = training_users.iloc[fit_idx]
            heldout = training_users.iloc[heldout_idx]

            assert set(fit["username"]).isdisjoint(heldout["username"])
            assert set(fit["username"]) | set(heldout["username"]) == train_set

            assignment[heldout_idx] = fold

            counts = (
                fit.groupby("label")["n_tweets"].sum()
                .reindex([0, 1], fill_value=0)
            )

            rows.append({
                "fold": fold,
                "fit_users": len(fit),
                "fit_nonASD": int(fit["label"].eq(0).sum()),
                "fit_ASD": int(fit["label"].eq(1).sum()),
                "heldout_users": len(heldout),
                "heldout_nonASD": int(heldout["label"].eq(0).sum()),
                "heldout_ASD": int(heldout["label"].eq(1).sum()),
                "fit_tweets_before_RUS": int(counts.sum()),
                "fit_tweets_after_RUS": int(2 * counts.min())
            })

        assert (assignment >= 1).all()
        training_users["heldout_fold"] = assignment

        summary = pd.DataFrame(rows)
        settings = {
            "status": "fold plan only; models not trained",
            "seed": seed,
            "n_folds": 3,
            "base_models": ["Random Forest", "TextCNN", "BERTweet"],
            "meta_model": "Logistic Regression",
            "meta_training": "training-tweet user-disjoint OOF probabilities",
            "validation_role": "threshold selection using user-level Macro-F1",
            "threshold_grid": {
                "start": 0.05, "stop": 0.95, "step": 0.01
            },
            "threshold_tie_rule": "closest to 0.50, then smaller threshold",
            "user_prediction": "majority vote; ties assigned ASD",
            "user_ROC_AUC": "mean tweet ASD probability",
            "test_role": "evaluation after settings are fixed",
            "base_model_selection_note":
                "Base models retained from the previous exploratory experiment; "
                "not selected anew using the held-out test."
        }


        for filename, frame in [
            ("PRIVATE_training_user_fold_plan.csv", training_users),
            ("fold_summary.csv", summary)
        ]:
            path = out / filename
            if path.is_file():
                pd.testing.assert_frame_equal(
                    pd.read_csv(path), frame, check_dtype=False
                )

        path = out / "experiment_plan.json"
        if path.is_file():
            assert json.loads(path.read_text()) == settings, (
                f'Fold plan | seed={seed} differs.'
            )

        plans.append((seed, out, training_users, summary, settings))


    for seed, out, training_users, summary, settings in plans:
        out.mkdir(parents=True, exist_ok=True)

        for filename, frame in [
            ("PRIVATE_training_user_fold_plan.csv", training_users),
            ("fold_summary.csv", summary)
        ]:
            path = out / filename
            if not path.exists():
                frame.to_csv(path, index=False)

        path = out / "experiment_plan.json"
        if not path.exists():
            path.write_text(json.dumps(settings, indent=2))

        print(f'Seed {seed}')
        print(summary.to_string(index=False))



### Fold Overview

Training and held-out users by class; tweet counts before and after undersampling.


In [8]:
show_result('OOF Fold Counts',saved_fold_overview)


OOF Fold Counts


,seed,fold,fit_users,fit_nonASD,fit_ASD,heldout_users,heldout_nonASD,heldout_ASD,fit_tweets_before_RUS,fit_tweets_after_RUS
0,42,1,97,12,85,49,6,43,375311,86030
1,42,2,97,12,85,49,6,43,350347,65246
2,42,3,98,12,86,48,6,42,357754,59060
3,43,1,97,12,85,49,6,43,375966,79504
4,43,2,97,12,85,49,6,43,356269,82012
5,43,3,98,12,86,48,6,42,357261,65320
6,44,1,97,12,85,49,6,43,386363,81658
7,44,2,97,12,85,49,6,43,393116,68498
8,44,3,98,12,86,48,6,42,369739,71924
9,45,1,97,12,85,49,6,43,371420,71644


,seed,fold,fit_users,fit_nonASD,fit_ASD,heldout_users,heldout_nonASD,heldout_ASD,fit_tweets_before_RUS,fit_tweets_after_RUS
0,42,1,97,12,85,49,6,43,375311,86030
1,42,2,97,12,85,49,6,43,350347,65246
2,42,3,98,12,86,48,6,42,357754,59060
3,43,1,97,12,85,49,6,43,375966,79504
4,43,2,97,12,85,49,6,43,356269,82012
5,43,3,98,12,86,48,6,42,357261,65320
6,44,1,97,12,85,49,6,43,386363,81658
7,44,2,97,12,85,49,6,43,393116,68498
8,44,3,98,12,86,48,6,42,369739,71924
9,45,1,97,12,85,49,6,43,371420,71644


## 3. RF and TF-IDF

10,000 TF-IDF features, unigrams and bigrams; 100 RF trees. Undersampling and fitting use only the training portion of each fold.


In [9]:
def train_rf_seed42():
    require_training('rf_42', meta=False)
    import gc
    import json
    import hashlib
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import sklearn
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.pipeline import make_pipeline

    BASE = Path("/content/drive_research/MyDrive/thesis/saved_results")
    OUT = BASE / "17_conference_revision/stacking_oof/seed_42"
    SPLITS = BASE / "17_conference_revision/multiseed/seed_42"

    data = pd.read_pickle(
        BASE / "02_preprocessing/cleaned_data.pkl"
    )[["username", "label", "cleaned_tweet"]].reset_index(drop=True)

    plan = pd.read_csv(OUT / "PRIVATE_training_user_fold_plan.csv")
    assert plan["username"].is_unique
    assert set(plan["heldout_fold"]) == {1, 2, 3}
    assert data["cleaned_tweet"].notna().all()
    assert data.groupby("username")["label"].nunique().eq(1).all()

    def get_positions(part):
        names = pd.read_csv(SPLITS / f"{part}_users.csv")["username"]
        return np.flatnonzero(data["username"].isin(names).to_numpy())

    train_positions = get_positions("train")
    val_positions = get_positions("val")
    test_positions = get_positions("test")

    assert set(data.iloc[train_positions]["username"]) == set(plan["username"])
    for left, right in [
        (train_positions, val_positions),
        (train_positions, test_positions),
        (val_positions, test_positions)
    ]:
        assert not set(data.iloc[left]["username"]) & set(
            data.iloc[right]["username"]
        )


    signature = hashlib.sha256(
        pd.util.hash_pandas_object(data, index=True).values.tobytes()
        + plan.to_csv(index=False).encode()
        + sklearn.__version__.encode()
        + b"RF100_TFIDF10000_ngram12_minDF2_sublinear_seed42_v1"
    ).hexdigest()

    def balanced_training(positions):
        part = data.iloc[positions]
        minority_n = part["label"].value_counts().min()

        return (
            pd.concat([
                group.sample(n=minority_n, random_state=42)
                if len(group) > minority_n else group
                for _, group in part.groupby("label")
            ])
            .sample(frac=1, random_state=42)
            .reset_index(drop=True)
        )

    def build_model():
        return make_pipeline(
            TfidfVectorizer(
                max_features=10000,
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True
            ),
            RandomForestClassifier(
                n_estimators=100,
                criterion="gini",
                max_depth=None,
                max_features="sqrt",
                min_samples_split=2,
                min_samples_leaf=1,
                bootstrap=True,
                class_weight=None,
                random_state=42,
                n_jobs=-1
            )
        )

    def load_cache(path):
        if not path.exists():
            return None
        with np.load(path, allow_pickle=False) as cached:
            assert cached["signature"].item() == signature, (
                f'Cache signature mismatch: {path.name}.'
            )
            return {key: cached[key].copy() for key in cached.files}


    oof_prob = np.full(len(data), np.nan)

    for fold in [1, 2, 3]:
        heldout_names = set(
            plan.loc[plan["heldout_fold"] == fold, "username"]
        )
        fit_names = set(
            plan.loc[plan["heldout_fold"] != fold, "username"]
        )
        assert fit_names.isdisjoint(heldout_names)

        fit_positions = np.flatnonzero(
            data["username"].isin(fit_names).to_numpy()
        )
        heldout_positions = np.flatnonzero(
            data["username"].isin(heldout_names).to_numpy()
        )

        path = OUT / f"PRIVATE_RF_fold_{fold}_probabilities.npz"
        cached = load_cache(path)

        if cached is not None:
            assert np.array_equal(
                cached["row_positions"], heldout_positions
            )
            probabilities = cached["probabilities"]
        else:
            balanced = balanced_training(fit_positions)
            model = build_model()

            print(f'RF | fold={fold}: {len(balanced):,} tweets', flush=True)
            model.fit(balanced["cleaned_tweet"], balanced["label"])

            positive_column = list(model.classes_).index(1)
            probabilities = model.predict_proba(
                data.iloc[heldout_positions]["cleaned_tweet"]
            )[:, positive_column]

            np.savez_compressed(
                path,
                signature=np.array(signature),
                row_positions=heldout_positions,
                probabilities=probabilities
            )
            del model, balanced
            gc.collect()

        assert np.isfinite(probabilities).all()
        assert len(probabilities) == len(heldout_positions)
        oof_prob[heldout_positions] = probabilities

    assert np.isfinite(oof_prob[train_positions]).all()

    np.savez_compressed(
        OUT / "PRIVATE_RF_OOF_probabilities.npz",
        signature=np.array(signature),
        row_positions=train_positions,
        probabilities=oof_prob[train_positions]
    )


    final_path = OUT / "PRIVATE_RF_final_probabilities.npz"
    cached = load_cache(final_path)

    if cached is None:
        balanced = balanced_training(train_positions)
        model = build_model()

        print('RF | seed=42 | final', flush=True)
        model.fit(balanced["cleaned_tweet"], balanced["label"])
        positive_column = list(model.classes_).index(1)

        val_prob = model.predict_proba(
            data.iloc[val_positions]["cleaned_tweet"]
        )[:, positive_column]
        test_prob = model.predict_proba(
            data.iloc[test_positions]["cleaned_tweet"]
        )[:, positive_column]

        assert np.isfinite(val_prob).all()
        assert np.isfinite(test_prob).all()

        np.savez_compressed(
            final_path,
            signature=np.array(signature),
            val_row_positions=val_positions,
            val_probabilities=val_prob,
            test_row_positions=test_positions,
            test_probabilities=test_prob
        )

        with open(OUT / "RF_parameters.json", "w") as f:
            json.dump({
                "signature": signature,
                "sklearn_version": sklearn.__version__,
                "parameters": model.get_params(deep=True),
                "undersampling": "fit subset only; random_state=42",
                "oof_folds": 3
            }, f, indent=2, default=str)

        del model, balanced
        gc.collect()
    else:
        assert np.array_equal(cached["val_row_positions"], val_positions)
        assert np.array_equal(cached["test_row_positions"], test_positions)



In [10]:
def train_rf_seed43_46():
    require_training('rf_43_46', meta=False)
    import gc
    import json
    import hashlib
    import os
    import tempfile
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import sklearn

    from sklearn.pipeline import Pipeline
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.ensemble import RandomForestClassifier

    BASE = Path("/content/drive_research/MyDrive/thesis/saved_results")
    ROOT = BASE / "17_conference_revision"

    data = pd.read_pickle(
        BASE / "02_preprocessing/cleaned_data.pkl"
    )[["username", "label", "cleaned_tweet"]].reset_index(drop=True)

    assert data.notna().all().all()
    assert data["label"].isin([0, 1]).all()
    assert data.groupby("username")["label"].nunique().eq(1).all()


    reference = json.loads(
        (ROOT / "stacking_oof/seed_42/RF_parameters.json").read_text()
    )
    assert sklearn.__version__ == reference["sklearn_version"], (
        f"scikit-learn version mismatch: current={sklearn.__version__} | seed=42 {reference['sklearn_version']}"
    )

    def balanced_positions(positions, seed):
        'Undersampling is restricted to the fitting subset.'
        frame = data.iloc[positions]
        minimum = int(frame["label"].value_counts().min())
        assert frame["label"].nunique() == 2

        pieces = []
        for _, group in frame.groupby("label"):
            if len(group) > minimum:
                group = group.sample(n=minimum, random_state=seed)
            pieces.append(group)

        balanced = pd.concat(pieces).sample(
            frac=1, random_state=seed
        )
        assert balanced["label"].value_counts().nunique() == 1
        return balanced.index.to_numpy()

    def make_rf(seed):
        return Pipeline([
            ("tfidf", TfidfVectorizer(
                max_features=10000,
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True
            )),
            ("rf", RandomForestClassifier(
                n_estimators=100,
                criterion="gini",
                max_depth=None,
                min_samples_split=2,
                min_samples_leaf=1,
                max_features="sqrt",
                bootstrap=True,
                class_weight=None,
                random_state=seed,
                n_jobs=-1
            ))
        ])

    def predict_asd(model, positions, batch_size=20000):
        positive_column = list(model.classes_).index(1)
        result = np.empty(len(positions), dtype=float)

        for start in range(0, len(positions), batch_size):
            selected = positions[start:start + batch_size]
            result[start:start + len(selected)] = model.predict_proba(
                data.iloc[selected]["cleaned_tweet"]
            )[:, positive_column]

        assert np.isfinite(result).all()
        assert ((result >= 0) & (result <= 1)).all()
        return result

    def save_npz(path, **arrays):

        assert not path.exists(), f'File exists: {path.name}'
        with tempfile.NamedTemporaryFile(
            dir=path.parent, suffix=".tmp", delete=False
        ) as handle:
            temporary = Path(handle.name)
            try:
                np.savez_compressed(handle, **arrays)
            except Exception:
                temporary.unlink(missing_ok=True)
                raise
        os.replace(temporary, path)

    def check_probability(values, positions):
        assert values.shape == (len(positions),)
        assert np.isfinite(values).all()
        assert ((values >= 0) & (values <= 1)).all()

    for seed in range(43, 47):
        print(f'RF | seed={seed}', flush=True)

        out = ROOT / f"stacking_oof/seed_{seed}"
        split_dir = ROOT / f"multiseed/seed_{seed}"
        assert out.is_dir()

        plan = pd.read_csv(
            out / "PRIVATE_training_user_fold_plan.csv"
        )
        assert plan["username"].is_unique
        assert plan["heldout_fold"].isin([1, 2, 3]).all()

        parts = {}
        for part in ["train", "val", "test"]:
            table = pd.read_csv(split_dir / f"{part}_users.csv")
            assert table["username"].notna().all()
            assert table["username"].is_unique
            parts[part] = set(table["username"])

        assert not (parts["train"] & parts["val"])
        assert not (parts["train"] & parts["test"])
        assert not (parts["val"] & parts["test"])
        assert set().union(*parts.values()) == set(data["username"])
        assert set(plan["username"]) == parts["train"]

        user_labels = data.groupby("username")["label"].first()
        assert np.array_equal(
            plan["label"].to_numpy(),
            user_labels.loc[plan["username"]].to_numpy()
        )

        positions = {
            part: np.flatnonzero(
                data["username"].isin(names).to_numpy()
            )
            for part, names in parts.items()
        }

        parameters = {
            "seed": seed,
            "tfidf": {
                "max_features": 10000,
                "ngram_range": [1, 2],
                "min_df": 2,
                "sublinear_tf": True
            },
            "rf": {
                "n_estimators": 100,
                "criterion": "gini",
                "max_depth": None,
                "min_samples_split": 2,
                "min_samples_leaf": 1,
                "max_features": "sqrt",
                "bootstrap": True,
                "class_weight": None,
                "random_state": seed,
                "n_jobs": -1
            },
            "undersampling": "fit portion only",
            "sklearn_version": sklearn.__version__
        }

        source_bytes = (
            pd.util.hash_pandas_object(data, index=True).values.tobytes()
            + plan.to_csv(index=False).encode()
            + json.dumps(
                {p: sorted(v) for p, v in parts.items()},
                sort_keys=True
            ).encode()
        )

        signature = hashlib.sha256(
            source_bytes
            + json.dumps(parameters, sort_keys=True).encode()
        ).hexdigest()

        config = {
            "signature": signature,
            "sklearn_version": sklearn.__version__,
            "parameters": parameters,
            "oof_folds": 3,
            "validation_used_in_base_training": False
        }

        config_path = out / "RF_parameters.json"
        if config_path.exists():
            assert json.loads(config_path.read_text()) == config, (
                'Configuration mismatch.'
            )
        else:
            config_path.write_text(json.dumps(config, indent=2))

        oof = np.full(len(data), np.nan)

        for fold in [1, 2, 3]:
            heldout_users = set(
                plan.loc[plan["heldout_fold"].eq(fold), "username"]
            )
            fit_users = parts["train"] - heldout_users

            assert fit_users.isdisjoint(heldout_users)

            fit_pos = np.flatnonzero(
                data["username"].isin(fit_users).to_numpy()
            )
            heldout_pos = np.flatnonzero(
                data["username"].isin(heldout_users).to_numpy()
            )

            path = out / f"PRIVATE_RF_fold_{fold}_probabilities.npz"

            if path.exists():
                with np.load(path, allow_pickle=False) as saved:
                    assert saved["signature"].item() == signature
                    assert np.array_equal(
                        saved["row_positions"], heldout_pos
                    )
                    probability = saved["probabilities"].copy()
                check_probability(probability, heldout_pos)

            else:
                balanced_pos = balanced_positions(fit_pos, seed)
                print(f'RF | fold={fold}: {len(balanced_pos):,} tweets', flush=True)

                model = make_rf(seed)
                model.fit(
                    data.iloc[balanced_pos]["cleaned_tweet"],
                    data.iloc[balanced_pos]["label"]
                )
                probability = predict_asd(model, heldout_pos)

                save_npz(
                    path,
                    signature=np.array(signature),
                    row_positions=heldout_pos,
                    probabilities=probability
                )
                del model
                gc.collect()

            assert np.isnan(oof[heldout_pos]).all()
            oof[heldout_pos] = probability

        train_pos = positions["train"]
        check_probability(oof[train_pos], train_pos)

        oof_path = out / "PRIVATE_RF_OOF_probabilities.npz"
        if oof_path.exists():
            with np.load(oof_path, allow_pickle=False) as saved:
                assert saved["signature"].item() == signature
                assert np.array_equal(saved["row_positions"], train_pos)
                assert np.array_equal(
                    saved["probabilities"], oof[train_pos]
                )
        else:
            save_npz(
                oof_path,
                signature=np.array(signature),
                row_positions=train_pos,
                probabilities=oof[train_pos]
            )

        final_path = out / "PRIVATE_RF_final_probabilities.npz"

        if final_path.exists():
            with np.load(final_path, allow_pickle=False) as saved:
                assert saved["signature"].item() == signature
                for part in ["val", "test"]:
                    assert np.array_equal(
                        saved[f"{part}_row_positions"], positions[part]
                    )
                    check_probability(
                        saved[f"{part}_probabilities"], positions[part]
                    )

        else:
            balanced_pos = balanced_positions(train_pos, seed)
            print(f'RF | final | n={len(balanced_pos):,} tweets', flush=True)

            model = make_rf(seed)
            model.fit(
                data.iloc[balanced_pos]["cleaned_tweet"],
                data.iloc[balanced_pos]["label"]
            )

            val_probability = predict_asd(model, positions["val"])
            test_probability = predict_asd(model, positions["test"])

            save_npz(
                final_path,
                signature=np.array(signature),
                val_row_positions=positions["val"],
                val_probabilities=val_probability,
                test_row_positions=positions["test"],
                test_probabilities=test_probability
            )
            del model
            gc.collect()

        del oof
        gc.collect()



## 4. TextCNN

10,000-word tokenizer, length 100, embedding dimension 64. Ten fixed epochs; save a model and tokenizer for each fold and the final refit.


In [11]:
def train_textcnn_seed42():
    require_training('textcnn_42', meta=False)
    import gc
    import json
    import hashlib
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import tensorflow as tf

    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import (
        Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
    )
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences

    BASE = Path("/content/drive_research/MyDrive/thesis/saved_results")
    OUT = BASE / "17_conference_revision/stacking_oof/seed_42"
    SPLITS = BASE / "17_conference_revision/multiseed/seed_42"

    assert (OUT / "PRIVATE_RF_OOF_probabilities.npz").is_file(), \
        'Missing RF probabilities; check BASE.'

    gpus = tf.config.list_physical_devices("GPU")
    assert gpus, 'TensorFlow GPU required.'

    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass

    tf.keras.mixed_precision.set_global_policy("float32")
    print('GPU TensorFlow:', gpus, flush=True)

    data = pd.read_pickle(
        BASE / "02_preprocessing/cleaned_data.pkl"
    )[["username", "label", "cleaned_tweet"]].reset_index(drop=True)

    plan = pd.read_csv(OUT / "PRIVATE_training_user_fold_plan.csv")
    assert plan["username"].is_unique
    assert set(plan["heldout_fold"]) == {1, 2, 3}
    assert data["cleaned_tweet"].notna().all()
    assert data.groupby("username")["label"].nunique().eq(1).all()

    def positions_for(part):
        names = pd.read_csv(SPLITS / f"{part}_users.csv")["username"]
        assert names.notna().all() and names.is_unique
        return np.flatnonzero(data["username"].isin(names).to_numpy())

    train_pos = positions_for("train")
    val_pos = positions_for("val")
    test_pos = positions_for("test")

    assert set(data.iloc[train_pos]["username"]) == set(plan["username"])
    for a, b in [(train_pos, val_pos), (train_pos, test_pos), (val_pos, test_pos)]:
        assert set(data.iloc[a]["username"]).isdisjoint(
            set(data.iloc[b]["username"])
        )


    data_plan_bytes = (
        pd.util.hash_pandas_object(data, index=True).values.tobytes()
        + plan.to_csv(index=False).encode()
    )

    with open(OUT / "RF_parameters.json") as f:
        rf_config = json.load(f)

    expected_rf_signature = hashlib.sha256(
        data_plan_bytes
        + rf_config["sklearn_version"].encode()
        + b"RF100_TFIDF10000_ngram12_minDF2_sublinear_seed42_v1"
    ).hexdigest()

    with np.load(
        OUT / "PRIVATE_RF_OOF_probabilities.npz", allow_pickle=False
    ) as rf:
        assert rf["signature"].item() == expected_rf_signature, \
            'Dataset or fold plan mismatch.'
        assert np.array_equal(rf["row_positions"], train_pos)

    PARAMETERS = {
        "seed": 42,
        "vocabulary_limit": 10000,
        "sequence_length": 100,
        "embedding_dimension": 64,
        "mask_zero": False,
        "padding": "post",
        "truncating": "post",
        "filters": 128,
        "kernel_size": 3,
        "convolution_padding": "valid",
        "dense_units": 64,
        "dropout": [0.3, 0.2],
        "optimizer": "Adam",
        "learning_rate": 0.001,
        "loss": "binary_crossentropy",
        "epochs": 10,
        "early_stopping": False,
        "train_batch_size": 64,
        "prediction_batch_size": 256,
        "tensorflow_version": tf.__version__,
        "keras_version": tf.keras.__version__,
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__
    }

    signature = hashlib.sha256(
        data_plan_bytes
        + json.dumps(PARAMETERS, sort_keys=True).encode()
    ).hexdigest()

    def balance_fit(positions):
        frame = data.iloc[positions]
        target = frame["label"].value_counts().min()
        return (
            pd.concat([
                group.sample(n=target, random_state=42)
                if len(group) > target else group
                for _, group in frame.groupby("label")
            ])
            .sample(frac=1, random_state=42)
            .reset_index(drop=True)
        )

    def make_sequences(tokenizer, texts):
        sequences = tokenizer.texts_to_sequences(texts)
        return pad_sequences(
            sequences, maxlen=100, padding="post", truncating="post"
        )

    def build_textcnn():
        return Sequential([
            Input(shape=(100,), dtype="int32"),
            Embedding(10000, 64, mask_zero=False),
            Conv1D(128, 3, activation="relu", padding="valid"),
            GlobalMaxPooling1D(),
            Dropout(0.3),
            Dense(64, activation="relu"),
            Dropout(0.2),
            Dense(1, activation="sigmoid")
        ])

    def read_cache(path):
        if not path.exists():
            return None
        with np.load(path, allow_pickle=False) as saved:
            assert saved["signature"].item() == signature, \
                f'Cache configuration mismatch: {path.name}'
            return {key: saved[key].copy() for key in saved.files}

    def train_textcnn(fit_positions, stage):
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(42)

        balanced = balance_fit(fit_positions)


        tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
        tokenizer.fit_on_texts(balanced["cleaned_tweet"])
        X_fit = make_sequences(tokenizer, balanced["cleaned_tweet"])
        y_fit = balanced["label"].to_numpy(dtype=np.float32)

        model = build_textcnn()
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
            loss="binary_crossentropy"
        )

        print(f'TextCNN | {stage}: {len(balanced):,} tweets | epochs=10', flush=True)

        history = model.fit(
            X_fit, y_fit,
            epochs=10,
            batch_size=64,
            shuffle=True,
            verbose=2,
            callbacks=[tf.keras.callbacks.TerminateOnNaN()]
        )

        assert len(history.history["loss"]) == 10, \
            'Training did not complete 10 epochs.'
        assert np.isfinite(history.history["loss"]).all(), \
            'Non-finite loss.'

        pd.DataFrame(history.history).to_csv(
            OUT / f"TextCNN_{stage}_history.csv", index=False
        )
        model.save(OUT / f"PRIVATE_TextCNN_{stage}.keras")

        with open(OUT / f"PRIVATE_TextCNN_{stage}_tokenizer.json", "w") as f:
            f.write(tokenizer.to_json())

        del balanced, X_fit, y_fit
        gc.collect()
        return model, tokenizer


    oof_prob = np.full(len(data), np.nan)

    for fold in [1, 2, 3]:
        held_names = set(
            plan.loc[plan["heldout_fold"] == fold, "username"]
        )
        fit_names = set(
            plan.loc[plan["heldout_fold"] != fold, "username"]
        )
        assert fit_names.isdisjoint(held_names)

        fit_pos = np.flatnonzero(data["username"].isin(fit_names).to_numpy())
        held_pos = np.flatnonzero(data["username"].isin(held_names).to_numpy())

        path = OUT / f"PRIVATE_TextCNN_fold_{fold}_probabilities.npz"
        cached = read_cache(path)

        if cached is not None:
            assert np.array_equal(cached["row_positions"], held_pos)
            probabilities = cached["probabilities"]
        else:
            model, tokenizer = train_textcnn(fit_pos, f"fold_{fold}")
            X_held = make_sequences(
                tokenizer, data.iloc[held_pos]["cleaned_tweet"]
            )
            probabilities = model.predict(
                X_held, batch_size=256, verbose=0
            ).reshape(-1)

            assert np.isfinite(probabilities).all()
            np.savez_compressed(
                path,
                signature=np.array(signature),
                row_positions=held_pos,
                probabilities=probabilities
            )

            del model, tokenizer, X_held
            tf.keras.backend.clear_session()
            gc.collect()

        assert len(probabilities) == len(held_pos)
        assert np.isfinite(probabilities).all()
        assert ((probabilities >= 0) & (probabilities <= 1)).all()
        oof_prob[held_pos] = probabilities

    assert np.isfinite(oof_prob[train_pos]).all()

    np.savez_compressed(
        OUT / "PRIVATE_TextCNN_OOF_probabilities.npz",
        signature=np.array(signature),
        row_positions=train_pos,
        probabilities=oof_prob[train_pos]
    )


    final_path = OUT / "PRIVATE_TextCNN_final_probabilities.npz"
    cached = read_cache(final_path)

    if cached is None:
        model, tokenizer = train_textcnn(train_pos, "final")

        final_outputs = {}
        for name, positions in [("val", val_pos), ("test", test_pos)]:
            X_part = make_sequences(
                tokenizer, data.iloc[positions]["cleaned_tweet"]
            )
            prob = model.predict(
                X_part, batch_size=256, verbose=0
            ).reshape(-1)
            assert np.isfinite(prob).all()
            final_outputs[f"{name}_row_positions"] = positions
            final_outputs[f"{name}_probabilities"] = prob
            del X_part

        np.savez_compressed(
            final_path,
            signature=np.array(signature),
            **final_outputs
        )
        del model, tokenizer
        tf.keras.backend.clear_session()
        gc.collect()
    else:
        assert np.array_equal(cached["val_row_positions"], val_pos)
        assert np.array_equal(cached["test_row_positions"], test_pos)

    with open(OUT / "TextCNN_parameters.json", "w") as f:
        json.dump({
            "signature": signature,
            "source_RF_signature": expected_rf_signature,
            "parameters": PARAMETERS,
            "undersampling": "fit subset only; random_state=42",
            "validation_used_in_base_training": False
        }, f, indent=2)



In [12]:
def train_textcnn_seed43_46():
    require_training('textcnn_43_46', meta=False)
    import gc
    import json
    import hashlib
    import os
    import tempfile
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import sklearn
    import tensorflow as tf

    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences

    BASE = Path("/content/drive_research/MyDrive/thesis/saved_results")
    ROOT = BASE / "17_conference_revision"

    assert tf.config.list_physical_devices("GPU"), 'TensorFlow GPU required.'

    reference = json.loads(
        (ROOT / "stacking_oof/seed_42/TextCNN_parameters.json").read_text()
    )
    reference_parameters = reference["parameters"]

    current_versions = {
        "tensorflow_version": tf.__version__,
        "keras_version": tf.keras.__version__,
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__
    }
    for key, value in current_versions.items():
        assert reference_parameters[key] == value, (
            f'{key} mismatch: seed42={reference_parameters[key]}, current={value}'
        )

    data = pd.read_pickle(
        BASE / "02_preprocessing/cleaned_data.pkl"
    )[["username", "label", "cleaned_tweet"]].reset_index(drop=True)

    assert data.notna().all().all()
    assert data["label"].isin([0, 1]).all()
    assert data.groupby("username")["label"].nunique().eq(1).all()

    data_bytes = pd.util.hash_pandas_object(
        data, index=True
    ).values.tobytes()

    def balanced_positions(positions, seed):
        frame = data.iloc[positions]
        assert frame["label"].nunique() == 2
        minimum = int(frame["label"].value_counts().min())

        pieces = []
        for _, group in frame.groupby("label"):
            if len(group) > minimum:
                group = group.sample(n=minimum, random_state=seed)
            pieces.append(group)

        balanced = pd.concat(pieces).sample(frac=1, random_state=seed)
        assert balanced["label"].value_counts().nunique() == 1
        return balanced.index.to_numpy()

    def save_probability_file(path, **arrays):
        assert not path.exists(), f'File exists: {path.name}'
        temporary = None
        try:
            with tempfile.NamedTemporaryFile(
                dir=path.parent, suffix=".tmp", delete=False
            ) as handle:
                temporary = Path(handle.name)
                np.savez_compressed(handle, **arrays)
            os.replace(temporary, path)
        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)

    def check_probability(values, positions):
        assert values.shape == (len(positions),)
        assert np.isfinite(values).all()
        assert ((values >= 0) & (values <= 1)).all()

    def encode(tokenizer, positions, parameters):
        sequences = tokenizer.texts_to_sequences(
            data.iloc[positions]["cleaned_tweet"].tolist()
        )
        return pad_sequences(
            sequences,
            maxlen=parameters["sequence_length"],
            padding=parameters["padding"],
            truncating=parameters["truncating"]
        )

    def fit_textcnn(fit_positions, seed, stage, out, parameters):
        tf.keras.backend.clear_session()
        tf.keras.mixed_precision.set_global_policy("float32")
        tf.keras.utils.set_random_seed(seed)
        gc.collect()

        selected = balanced_positions(fit_positions, seed)

        tokenizer = Tokenizer(
            num_words=parameters["vocabulary_limit"],
            oov_token="<OOV>"
        )
        tokenizer.fit_on_texts(
            data.iloc[selected]["cleaned_tweet"].tolist()
        )

        X = encode(tokenizer, selected, parameters)
        y = data.iloc[selected]["label"].to_numpy(dtype=np.float32)

        model = tf.keras.Sequential([
            tf.keras.layers.Input(
                shape=(parameters["sequence_length"],),
                dtype="int32"
            ),
            tf.keras.layers.Embedding(
                input_dim=parameters["vocabulary_limit"],
                output_dim=parameters["embedding_dimension"],
                mask_zero=parameters["mask_zero"]
            ),
            tf.keras.layers.Conv1D(
                filters=parameters["filters"],
                kernel_size=parameters["kernel_size"],
                activation="relu",
                padding=parameters["convolution_padding"]
            ),
            tf.keras.layers.GlobalMaxPooling1D(),
            tf.keras.layers.Dropout(parameters["dropout"][0]),
            tf.keras.layers.Dense(
                parameters["dense_units"], activation="relu"
            ),
            tf.keras.layers.Dropout(parameters["dropout"][1]),
            tf.keras.layers.Dense(1, activation="sigmoid")
        ])

        model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=parameters["learning_rate"]
            ),
            loss=parameters["loss"]
        )

        print(f'TextCNN | seed={seed}, {stage}: {len(selected):,} tweets | epochs=10', flush=True)

        history = model.fit(
            X, y,
            epochs=parameters["epochs"],
            batch_size=parameters["train_batch_size"],
            shuffle=True,
            verbose=2
        )

        pd.DataFrame(history.history).to_csv(
            out / f"TextCNN_{stage}_history.csv", index=False
        )
        model.save(out / f"PRIVATE_TextCNN_{stage}.keras")
        (out / f"PRIVATE_TextCNN_{stage}_tokenizer.json").write_text(
            tokenizer.to_json()
        )

        del X, y
        gc.collect()
        return model, tokenizer

    def predict_textcnn(model, tokenizer, positions, parameters):
        X = encode(tokenizer, positions, parameters)
        probability = model.predict(
            X,
            batch_size=parameters["prediction_batch_size"],
            verbose=0
        ).reshape(-1)

        check_probability(probability, positions)
        del X
        gc.collect()
        return probability

    for seed in range(43, 47):
        print(f'TextCNN | seed={seed}', flush=True)

        out = ROOT / f"stacking_oof/seed_{seed}"
        split_dir = ROOT / f"multiseed/seed_{seed}"
        assert out.is_dir()

        plan = pd.read_csv(
            out / "PRIVATE_training_user_fold_plan.csv"
        )
        assert plan["username"].is_unique
        assert plan["heldout_fold"].isin([1, 2, 3]).all()

        parts = {}
        for part in ["train", "val", "test"]:
            table = pd.read_csv(split_dir / f"{part}_users.csv")
            assert table["username"].notna().all()
            assert table["username"].is_unique
            parts[part] = set(table["username"])

        assert not (parts["train"] & parts["val"])
        assert not (parts["train"] & parts["test"])
        assert not (parts["val"] & parts["test"])
        assert set().union(*parts.values()) == set(data["username"])
        assert set(plan["username"]) == parts["train"]

        actual_labels = data.groupby("username")["label"].first()
        assert np.array_equal(
            plan["label"].to_numpy(),
            actual_labels.loc[plan["username"]].to_numpy()
        )

        positions = {
            part: np.flatnonzero(
                data["username"].isin(names).to_numpy()
            )
            for part, names in parts.items()
        }

        source_bytes = (
            data_bytes
            + plan.to_csv(index=False).encode()
            + json.dumps(
                {part: sorted(names) for part, names in parts.items()},
                sort_keys=True
            ).encode()
        )


        rf_config = json.loads((out / "RF_parameters.json").read_text())
        assert rf_config["sklearn_version"] == sklearn.__version__

        expected_rf_signature = hashlib.sha256(
            source_bytes
            + json.dumps(
                rf_config["parameters"], sort_keys=True
            ).encode()
        ).hexdigest()
        assert rf_config["signature"] == expected_rf_signature

        for filename in [
            "PRIVATE_RF_OOF_probabilities.npz",
            "PRIVATE_RF_final_probabilities.npz"
        ]:
            with np.load(out / filename, allow_pickle=False) as saved:
                assert saved["signature"].item() == expected_rf_signature

        parameters = dict(reference_parameters)
        parameters["seed"] = seed

        tokenizer_settings = {
            "num_words": parameters["vocabulary_limit"],
            "oov_token": "<OOV>"
        }

        signature = hashlib.sha256(
            source_bytes
            + expected_rf_signature.encode()
            + json.dumps(parameters, sort_keys=True).encode()
            + json.dumps(tokenizer_settings, sort_keys=True).encode()
        ).hexdigest()

        config = {
            "signature": signature,
            "source_RF_signature": expected_rf_signature,
            "parameters": parameters,
            "tokenizer_settings": tokenizer_settings,
            "undersampling": "fit portion only",
            "validation_used_in_base_training": False
        }

        config_path = out / "TextCNN_parameters.json"
        if config_path.exists():
            assert json.loads(config_path.read_text()) == config, (
                'TextCNN configuration mismatch.'
            )
        else:
            config_path.write_text(json.dumps(config, indent=2))

        oof = np.full(len(data), np.nan)

        for fold in [1, 2, 3]:
            heldout_users = set(
                plan.loc[plan["heldout_fold"].eq(fold), "username"]
            )
            fit_users = parts["train"] - heldout_users
            assert fit_users.isdisjoint(heldout_users)

            fit_pos = np.flatnonzero(
                data["username"].isin(fit_users).to_numpy()
            )
            heldout_pos = np.flatnonzero(
                data["username"].isin(heldout_users).to_numpy()
            )

            path = out / f"PRIVATE_TextCNN_fold_{fold}_probabilities.npz"

            if path.exists():
                with np.load(path, allow_pickle=False) as saved:
                    assert saved["signature"].item() == signature
                    assert np.array_equal(
                        saved["row_positions"], heldout_pos
                    )
                    probability = saved["probabilities"].copy()
                check_probability(probability, heldout_pos)

            else:
                model, tokenizer = fit_textcnn(
                    fit_pos, seed, f"fold_{fold}", out, parameters
                )
                probability = predict_textcnn(
                    model, tokenizer, heldout_pos, parameters
                )

                save_probability_file(
                    path,
                    signature=np.array(signature),
                    row_positions=heldout_pos,
                    probabilities=probability
                )

                del model, tokenizer
                tf.keras.backend.clear_session()
                gc.collect()

            assert np.isnan(oof[heldout_pos]).all()
            oof[heldout_pos] = probability

        train_pos = positions["train"]
        check_probability(oof[train_pos], train_pos)

        oof_path = out / "PRIVATE_TextCNN_OOF_probabilities.npz"
        if oof_path.exists():
            with np.load(oof_path, allow_pickle=False) as saved:
                assert saved["signature"].item() == signature
                assert np.array_equal(saved["row_positions"], train_pos)
                assert np.array_equal(
                    saved["probabilities"], oof[train_pos]
                )
        else:
            save_probability_file(
                oof_path,
                signature=np.array(signature),
                row_positions=train_pos,
                probabilities=oof[train_pos]
            )

        final_path = out / "PRIVATE_TextCNN_final_probabilities.npz"

        if final_path.exists():
            with np.load(final_path, allow_pickle=False) as saved:
                assert saved["signature"].item() == signature
                for part in ["val", "test"]:
                    assert np.array_equal(
                        saved[f"{part}_row_positions"], positions[part]
                    )
                    check_probability(
                        saved[f"{part}_probabilities"], positions[part]
                    )

        else:
            model, tokenizer = fit_textcnn(
                train_pos, seed, "final", out, parameters
            )

            val_probability = predict_textcnn(
                model, tokenizer, positions["val"], parameters
            )
            test_probability = predict_textcnn(
                model, tokenizer, positions["test"], parameters
            )

            save_probability_file(
                final_path,
                signature=np.array(signature),
                val_row_positions=positions["val"],
                val_probabilities=val_probability,
                test_row_positions=positions["test"],
                test_probabilities=test_probability
            )

            del model, tokenizer
            tf.keras.backend.clear_session()
            gc.collect()

        del oof
        gc.collect()



## 5. BERTweet

Length 128, fixed padding, normalization=False; two fixed epochs, AdamW 2e-5. The pretrained revision follows the experiment configuration.


In [13]:
def train_bertweet_seed42():
    require_training('bertweet_42', meta=False)
    import os
    os.environ["USE_TF"] = "0"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    import gc
    import json
    import random
    import hashlib
    from pathlib import Path
    import importlib.metadata as metadata

    import numpy as np
    import pandas as pd
    import torch
    from torch.utils.data import Dataset, DataLoader
    from tqdm.auto import tqdm

    from transformers import (
        AutoConfig,
        AutoTokenizer,
        AutoModelForSequenceClassification,
        get_linear_schedule_with_warmup
    )

    BASE = Path("/content/drive_research/MyDrive/thesis/saved_results")
    OUT = BASE / "17_conference_revision/stacking_oof/seed_42"
    SPLITS = BASE / "17_conference_revision/multiseed/seed_42"

    assert torch.cuda.is_available(), 'GPU required.'
    DEVICE = torch.device("cuda")

    data = pd.read_pickle(
        BASE / "02_preprocessing/cleaned_data.pkl"
    )[["username", "label", "cleaned_tweet"]].reset_index(drop=True)

    plan = pd.read_csv(OUT / "PRIVATE_training_user_fold_plan.csv")
    assert plan["username"].is_unique
    assert set(plan["heldout_fold"]) == {1, 2, 3}
    assert data["cleaned_tweet"].notna().all()
    assert data.groupby("username")["label"].nunique().eq(1).all()

    def positions_for(part):
        names = pd.read_csv(SPLITS / f"{part}_users.csv")["username"]
        assert names.notna().all() and names.is_unique
        return np.flatnonzero(data["username"].isin(names).to_numpy())

    train_pos = positions_for("train")
    val_pos = positions_for("val")
    test_pos = positions_for("test")

    assert set(data.iloc[train_pos]["username"]) == set(plan["username"])
    for a, b in [(train_pos, val_pos), (train_pos, test_pos), (val_pos, test_pos)]:
        assert set(data.iloc[a]["username"]).isdisjoint(
            set(data.iloc[b]["username"])
        )


    data_plan_bytes = (
        pd.util.hash_pandas_object(data, index=True).values.tobytes()
        + plan.to_csv(index=False).encode()
    )

    with open(OUT / "RF_parameters.json") as f:
        rf_config = json.load(f)

    rf_signature = hashlib.sha256(
        data_plan_bytes
        + rf_config["sklearn_version"].encode()
        + b"RF100_TFIDF10000_ngram12_minDF2_sublinear_seed42_v1"
    ).hexdigest()

    with np.load(
        OUT / "PRIVATE_RF_OOF_probabilities.npz", allow_pickle=False
    ) as saved:
        assert saved["signature"].item() == rf_signature
        assert np.array_equal(saved["row_positions"], train_pos)

    with open(OUT / "TextCNN_parameters.json") as f:
        cnn_config = json.load(f)

    assert cnn_config["source_RF_signature"] == rf_signature

    with np.load(
        OUT / "PRIVATE_TextCNN_OOF_probabilities.npz", allow_pickle=False
    ) as saved:
        assert saved["signature"].item() == cnn_config["signature"]
        assert np.array_equal(saved["row_positions"], train_pos)


    HF_ID = "vinai/bertweet-base"
    PARAM_PATH = OUT / "BERTweet_parameters.json"

    if PARAM_PATH.exists():
        with open(PARAM_PATH) as f:
            previous = json.load(f)
        REVISION = previous["parameters"]["pretrained_revision"]
    else:
        pretrained_config = AutoConfig.from_pretrained(HF_ID)
        REVISION = getattr(pretrained_config, "_commit_hash", None)
        assert REVISION, 'Pretrained revision is missing.'

    versions = {
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "numpy": np.__version__,
        "pandas": pd.__version__
    }
    for package in ["transformers", "sentencepiece", "emoji"]:
        versions[package] = metadata.version(package)

    PARAMETERS = {
        "seed": 42,
        "hf_id": HF_ID,
        "pretrained_revision": REVISION,
        "tokenizer_normalization": False,
        "max_length": 128,
        "epochs": 2,
        "train_batch_size": 32,
        "prediction_batch_size": 64,
        "optimizer": "torch.optim.AdamW",
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_fraction": 0.10,
        "mixed_precision": "float16",
        "checkpoint_selection": "fixed final epoch 2",
        "validation_used_in_base_training": False,
        "versions": versions
    }

    signature = hashlib.sha256(
        data_plan_bytes
        + json.dumps(PARAMETERS, sort_keys=True).encode()
    ).hexdigest()

    if PARAM_PATH.exists():
        assert previous["signature"] == signature, (
            'BERTweet configuration, data, or version mismatch.'
        )

    with open(PARAM_PATH, "w") as f:
        json.dump({
            "signature": signature,
            "source_RF_signature": rf_signature,
            "parameters": PARAMETERS,
            "undersampling": "fit subset only; random_state=42"
        }, f, indent=2)

    def load_cache(path):
        if not path.exists():
            return None
        with np.load(path, allow_pickle=False) as saved:
            assert saved["signature"].item() == signature, \
                f'Cache signature mismatch: {path.name}'
            return {key: saved[key].copy() for key in saved.files}



    tokenizer = AutoTokenizer.from_pretrained(
        HF_ID,
        revision=REVISION,
        use_fast=True,
        normalization=False
    )

    print('GPU:', torch.cuda.get_device_name(0))
    print('BERTweet revision:', REVISION)

    input_ids = np.empty((len(data), 128), dtype=np.int32)
    attention_masks = np.empty((len(data), 128), dtype=np.uint8)

    for start in tqdm(range(0, len(data), 2048), desc='Tokenization'):
        stop = min(start + 2048, len(data))
        encoded = tokenizer(
            data["cleaned_tweet"].iloc[start:stop].tolist(),
            truncation=True,
            max_length=128,
            padding="max_length",
            return_tensors="np"
        )
        input_ids[start:stop] = encoded["input_ids"]
        attention_masks[start:stop] = encoded["attention_mask"]

    labels = data["label"].to_numpy(dtype=np.int64)

    class TweetDataset(Dataset):
        def __init__(self, positions):
            self.positions = np.asarray(positions, dtype=np.int64)

        def __len__(self):
            return len(self.positions)

        def __getitem__(self, index):
            position = self.positions[index]
            return {
                "input_ids": torch.tensor(
                    input_ids[position], dtype=torch.long
                ),
                "attention_mask": torch.tensor(
                    attention_masks[position], dtype=torch.long
                ),
                "labels": torch.tensor(labels[position], dtype=torch.long)
            }

    def balanced_positions(positions):
        frame = data.iloc[positions]
        target = frame["label"].value_counts().min()
        sampled = pd.concat([
            group.sample(n=target, random_state=42)
            if len(group) > target else group
            for _, group in frame.groupby("label")
        ]).sample(frac=1, random_state=42)

        return sampled.index.to_numpy(dtype=np.int64)

    def train_bertweet(fit_positions, stage):
        random.seed(42)
        np.random.seed(42)
        torch.manual_seed(42)
        torch.cuda.manual_seed_all(42)

        balanced = balanced_positions(fit_positions)
        generator = torch.Generator().manual_seed(42)

        loader = DataLoader(
            TweetDataset(balanced),
            batch_size=32,
            shuffle=True,
            generator=generator,
            num_workers=0,
            pin_memory=True
        )

        model = AutoModelForSequenceClassification.from_pretrained(
            HF_ID, revision=REVISION, num_labels=2
        ).to(DEVICE)

        optimizer = torch.optim.AdamW(
            model.parameters(), lr=2e-5, weight_decay=0.01
        )
        total_steps = len(loader) * 2
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(0.10 * total_steps),
            num_training_steps=total_steps
        )
        scaler = torch.amp.GradScaler("cuda")

        print(f'BERTweet | {stage}: {len(balanced):,} tweets | epochs=2', flush=True)

        history = []

        for epoch in range(1, 3):
            model.train()
            loss_sum, n_examples = 0.0, 0

            for batch in tqdm(loader, desc=f"{stage}, epoch {epoch}/2"):
                batch = {
                    key: value.to(DEVICE, non_blocking=True)
                    for key, value in batch.items()
                }
                optimizer.zero_grad(set_to_none=True)

                with torch.autocast(
                    device_type="cuda", dtype=torch.float16
                ):
                    loss = model(**batch).loss

                assert torch.isfinite(loss).item(), 'Non-finite loss.'

                scaler.scale(loss).backward()

                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()


                if scaler.get_scale() >= previous_scale:
                    scheduler.step()

                size = batch["labels"].shape[0]
                loss_sum += float(loss.detach().cpu()) * size
                n_examples += size

            mean_loss = loss_sum / n_examples
            history.append({"epoch": epoch, "training_loss": mean_loss})

            pd.DataFrame(history).to_csv(
                OUT / f"BERTweet_{stage}_history.csv", index=False
            )
            print(f'{stage}, epoch {epoch}: loss={mean_loss:.4f}', flush=True)

        del optimizer, scheduler, scaler, loader
        gc.collect()
        torch.cuda.empty_cache()
        return model

    def predict_probabilities(model, positions, description):
        loader = DataLoader(
            TweetDataset(positions),
            batch_size=64,
            shuffle=False,
            num_workers=0,
            pin_memory=True
        )

        model.eval()
        probabilities = []

        with torch.inference_mode():
            for batch in tqdm(loader, desc=description):
                batch.pop("labels")
                batch = {
                    key: value.to(DEVICE, non_blocking=True)
                    for key, value in batch.items()
                }

                with torch.autocast(
                    device_type="cuda", dtype=torch.float16
                ):
                    logits = model(**batch).logits

                probabilities.append(
                    torch.softmax(logits.float(), dim=1)[:, 1]
                    .cpu().numpy()
                )

        result = np.concatenate(probabilities)
        assert len(result) == len(positions)
        assert np.isfinite(result).all()
        assert ((result >= 0) & (result <= 1)).all()
        return result


    oof_prob = np.full(len(data), np.nan)

    for fold in [1, 2, 3]:
        held_names = set(
            plan.loc[plan["heldout_fold"] == fold, "username"]
        )
        fit_names = set(
            plan.loc[plan["heldout_fold"] != fold, "username"]
        )
        assert fit_names.isdisjoint(held_names)

        fit_pos = np.flatnonzero(data["username"].isin(fit_names).to_numpy())
        held_pos = np.flatnonzero(data["username"].isin(held_names).to_numpy())

        path = OUT / f"PRIVATE_BERTweet_fold_{fold}_probabilities.npz"
        cached = load_cache(path)

        if cached is not None:
            assert np.array_equal(cached["row_positions"], held_pos)
            probabilities = cached["probabilities"]
        else:
            model = train_bertweet(fit_pos, f"fold_{fold}")
            probabilities = predict_probabilities(
                model, held_pos, f'OOF fold {fold}'
            )

            np.savez_compressed(
                path,
                signature=np.array(signature),
                row_positions=held_pos,
                probabilities=probabilities
            )

            del model
            gc.collect()
            torch.cuda.empty_cache()

        assert np.isfinite(probabilities).all()
        oof_prob[held_pos] = probabilities

    assert np.isfinite(oof_prob[train_pos]).all()

    np.savez_compressed(
        OUT / "PRIVATE_BERTweet_OOF_probabilities.npz",
        signature=np.array(signature),
        row_positions=train_pos,
        probabilities=oof_prob[train_pos]
    )


    final_path = OUT / "PRIVATE_BERTweet_final_probabilities.npz"
    cached = load_cache(final_path)

    if cached is None:
        model = train_bertweet(train_pos, "final")

        val_prob = predict_probabilities(
            model, val_pos, 'Validation'
        )
        test_prob = predict_probabilities(
            model, test_pos, 'Test'
        )

        np.savez_compressed(
            final_path,
            signature=np.array(signature),
            val_row_positions=val_pos,
            val_probabilities=val_prob,
            test_row_positions=test_pos,
            test_probabilities=test_prob
        )

        del model
        gc.collect()
        torch.cuda.empty_cache()
    else:
        assert np.array_equal(cached["val_row_positions"], val_pos)
        assert np.array_equal(cached["test_row_positions"], test_pos)



In [14]:
def train_bertweet_seed43_46():
    require_training('bertweet_43_46', meta=False)
    import os
    os.environ['USE_TF'] = '0'
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'

    import gc
    import json
    import hashlib
    import tempfile
    import random
    import platform
    import importlib.metadata as metadata
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import sklearn
    import torch
    import transformers
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from transformers import get_linear_schedule_with_warmup
    from tqdm.auto import tqdm

    assert transformers.__version__ == '4.57.1', (
        'Transformers 4.57.1 required; restart the kernel after installation.'
    )
    assert torch.cuda.is_available(), 'GPU required.'
    DEVICE = torch.device('cuda')
    BASE = Path('/content/drive_research/MyDrive/thesis/saved_results')
    if not (BASE / '02_preprocessing/cleaned_data.pkl').is_file():
        from google.colab import drive
        drive.mount('/content/drive_research')
    ROOT = BASE / '17_conference_revision'
    reference = json.loads(
        (ROOT / 'stacking_oof/seed_42/BERTweet_parameters.json').read_text()
    )['parameters']
    versions = {
        'torch': torch.__version__, 'cuda': torch.version.cuda,
        'numpy': np.__version__, 'pandas': pd.__version__,
        'transformers': transformers.__version__,
        'sentencepiece': metadata.version('sentencepiece'),
        'emoji': metadata.version('emoji')
    }
    assert versions == reference['versions'], ('Package versions must match seed 42.', {'seed42': reference['versions'], 'current': versions})
    assert reference['hf_id'] == 'vinai/bertweet-base'
    assert reference['pretrained_revision'] == 'b349c1243407b0dcffeabb2337497477286e27ab'
    assert reference['epochs'] == 2
    assert reference['checkpoint_selection'] == 'fixed final epoch 2'
    assert reference['validation_used_in_base_training'] is False
    print('GPU:', torch.cuda.get_device_name(0), flush=True)

    data = pd.read_pickle(BASE / '02_preprocessing/cleaned_data.pkl')[
        ['username', 'label', 'cleaned_tweet']
    ].reset_index(drop=True)
    assert data.notna().all().all()
    assert data['label'].isin([0, 1]).all()
    assert data.groupby('username')['label'].nunique().eq(1).all()
    assert data['username'].nunique() == 244
    labels = data['label'].to_numpy(dtype=np.int64)
    user_table = data.groupby('username', sort=True).agg(
        label=('label', 'first'), n_tweets=('label', 'size')
    )
    data_bytes = pd.util.hash_pandas_object(data, index=True).values.tobytes()

    def balanced_positions(positions, seed):
        frame = data.iloc[positions]
        assert frame['label'].nunique() == 2
        minimum = int(frame['label'].value_counts().min())
        pieces = []
        for _, group in frame.groupby('label'):
            if len(group) > minimum:
                group = group.sample(n=minimum, random_state=seed)
            pieces.append(group)
        balanced = pd.concat(pieces).sample(frac=1, random_state=seed)
        assert balanced['label'].value_counts().nunique() == 1
        return balanced.index.to_numpy()

    def check_probability(values, positions):
        assert values.shape == (len(positions),)
        assert np.isfinite(values).all()
        assert ((values >= 0) & (values <= 1)).all()

    def save_npz(path, **arrays):
        assert not path.exists(), f'File exists: {path.name}'
        temporary = None
        try:
            with tempfile.NamedTemporaryFile(
                dir=path.parent, suffix='.tmp', delete=False
            ) as handle:
                temporary = Path(handle.name)
                np.savez_compressed(handle, **arrays)
            os.replace(temporary, path)
        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)

    def read_probabilities(path, signature, positions, prefix=None):
        with np.load(path, allow_pickle=False) as saved:
            assert saved['signature'].item() == signature, path.name
            row_key = 'row_positions' if prefix is None else f'{prefix}_row_positions'
            probability_key = 'probabilities' if prefix is None else f'{prefix}_probabilities'
            assert np.array_equal(saved[row_key], positions), path.name
            values = saved[probability_key].copy()
        check_probability(values, positions)
        return values


    experiments = []
    for seed in range(43, 47):
        out = ROOT / f'stacking_oof/seed_{seed}'
        split_dir = ROOT / f'multiseed/seed_{seed}'
        assert out.is_dir()
        plan = pd.read_csv(out / 'PRIVATE_training_user_fold_plan.csv')
        assert plan['username'].is_unique
        assert plan['heldout_fold'].isin([1, 2, 3]).all()
        parts = {}
        for part in ['train', 'val', 'test']:
            table = pd.read_csv(split_dir / f'{part}_users.csv')
            assert table['username'].notna().all() and table['username'].is_unique
            parts[part] = set(table['username'])
        assert not (parts['train'] & parts['val'])
        assert not (parts['train'] & parts['test'])
        assert not (parts['val'] & parts['test'])
        assert set().union(*parts.values()) == set(data['username'])
        assert tuple(len(parts[p]) for p in ['train', 'val', 'test']) == (146, 37, 61)
        assert set(plan['username']) == parts['train']
        expected_users = user_table.loc[plan['username']]
        assert np.array_equal(plan['label'], expected_users['label'])
        assert np.array_equal(plan['n_tweets'], expected_users['n_tweets'])
        positions = {
            part: np.flatnonzero(data['username'].isin(names).to_numpy())
            for part, names in parts.items()
        }
        source_bytes = (
            data_bytes + plan.to_csv(index=False).encode()
            + json.dumps({p: sorted(v) for p, v in parts.items()}, sort_keys=True).encode()
        )
        rf_config = json.loads((out / 'RF_parameters.json').read_text())
        assert rf_config['sklearn_version'] == sklearn.__version__
        rf_signature = hashlib.sha256(
            source_bytes + json.dumps(rf_config['parameters'], sort_keys=True).encode()
        ).hexdigest()
        assert rf_config['signature'] == rf_signature
        cnn_config = json.loads((out / 'TextCNN_parameters.json').read_text())
        assert cnn_config['source_RF_signature'] == rf_signature
        expected_cnn_signature = hashlib.sha256(
            source_bytes + rf_signature.encode()
            + json.dumps(cnn_config['parameters'], sort_keys=True).encode()
            + json.dumps(cnn_config['tokenizer_settings'], sort_keys=True).encode()
        ).hexdigest()
        assert cnn_config['signature'] == expected_cnn_signature
        for name, signature in [('RF', rf_signature), ('TextCNN', expected_cnn_signature)]:
            read_probabilities(out / f'PRIVATE_{name}_OOF_probabilities.npz', signature, positions['train'])
            for part in ['val', 'test']:
                read_probabilities(out / f'PRIVATE_{name}_final_probabilities.npz', signature, positions[part], part)
        parameters = json.loads(json.dumps(reference))
        parameters['seed'] = seed
        config = {
            'source_RF_signature': rf_signature,
            'parameters': parameters,
            'python_version': platform.python_version(),
            'tokenization': 'fixed max-length padding; pretrained tokenizer; no fitted vocabulary',
            'undersampling': 'fit portion only',
            'validation_used_in_base_training': False
        }
        signature = hashlib.sha256(
            source_bytes + json.dumps(config, sort_keys=True).encode()
        ).hexdigest()
        config['signature'] = signature
        config_path = out / 'BERTweet_parameters.json'
        if config_path.exists():
            assert json.loads(config_path.read_text()) == config, (
                f'BERTweet configuration | seed={seed} changed.'
            )
        folds = []
        for fold in [1, 2, 3]:
            heldout_users = set(plan.loc[plan['heldout_fold'].eq(fold), 'username'])
            assert heldout_users
            fit_users = parts['train'] - heldout_users
            fit_pos = np.flatnonzero(data['username'].isin(fit_users).to_numpy())
            heldout_pos = np.flatnonzero(data['username'].isin(heldout_users).to_numpy())
            path = out / f'PRIVATE_BERTweet_fold_{fold}_probabilities.npz'
            if path.exists():
                read_probabilities(path, signature, heldout_pos)
            folds.append((fold, fit_pos, heldout_pos, path))
        oof_path = out / 'PRIVATE_BERTweet_OOF_probabilities.npz'
        if oof_path.exists():
            read_probabilities(oof_path, signature, positions['train'])
            assert all(item[3].is_file() for item in folds), 'OOF cache exists but fold caches are incomplete.'
        final_path = out / 'PRIVATE_BERTweet_final_probabilities.npz'
        if final_path.exists():
            for part in ['val', 'test']:
                read_probabilities(final_path, signature, positions[part], part)
        experiments.append((seed, out, parameters, config, positions, folds))

    needs_tokens = any(
        not all(item[3].is_file() for item in folds)
        or not (out / 'PRIVATE_BERTweet_final_probabilities.npz').is_file()
        for _, out, _, _, _, folds in experiments
    )
    if needs_tokens:
        tokenizer = AutoTokenizer.from_pretrained(
            reference['hf_id'], revision=reference['pretrained_revision'],
            use_fast=True, normalization=reference['tokenizer_normalization']
        )
        assert getattr(tokenizer, 'normalization', None) is False
        length = reference['max_length']
        input_ids = np.empty((len(data), length), dtype=np.int32)
        attention_mask = np.empty((len(data), length), dtype=np.uint8)
        for start in tqdm(range(0, len(data), 2048), desc='Tokenization'):
            texts = data['cleaned_tweet'].iloc[start:start + 2048].tolist()
            tokens = tokenizer(texts, padding='max_length', truncation=True, max_length=length)
            end = start + len(texts)
            input_ids[start:end] = np.asarray(tokens['input_ids'], dtype=np.int32)
            attention_mask[start:end] = np.asarray(tokens['attention_mask'], dtype=np.uint8)
        del tokens, tokenizer
        gc.collect()

    class EncodedTweets(Dataset):
        def __init__(self, positions):
            self.positions = np.asarray(positions)
        def __len__(self):
            return len(self.positions)
        def __getitem__(self, i):
            row = self.positions[i]
            return {
                'input_ids': torch.tensor(input_ids[row], dtype=torch.long),
                'attention_mask': torch.tensor(attention_mask[row], dtype=torch.long),
                'labels': torch.tensor(labels[row], dtype=torch.long)
            }

    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    def fit_bertweet(fit_positions, seed, stage, out, parameters):
        gc.collect()
        torch.cuda.empty_cache()
        set_seed(seed)
        selected = balanced_positions(fit_positions, seed)
        model = AutoModelForSequenceClassification.from_pretrained(
            parameters['hf_id'], revision=parameters['pretrained_revision'], num_labels=2
        ).to(DEVICE)
        loader = DataLoader(
            EncodedTweets(selected), batch_size=parameters['train_batch_size'],
            shuffle=True, generator=torch.Generator().manual_seed(seed), num_workers=0
        )
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=parameters['learning_rate'], weight_decay=parameters['weight_decay']
        )
        total_steps = len(loader) * parameters['epochs']
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=int(total_steps * parameters['warmup_fraction']),
            num_training_steps=total_steps
        )
        scaler = torch.amp.GradScaler('cuda')
        history = []
        print(f'BERTweet | seed={seed}, {stage}: {len(selected):,} tweets | epochs=2', flush=True)
        for epoch in range(parameters['epochs']):
            model.train()
            total_loss = 0.0
            for batch in tqdm(loader, desc=f'Seed {seed}, {stage}, epoch {epoch + 1}/2'):
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    loss = model(**batch).loss
                assert torch.isfinite(loss).item(), 'Non-finite loss.'
                scaler.scale(loss).backward()
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= old_scale:
                    scheduler.step()
                total_loss += float(loss.detach())
            avg_loss = total_loss / len(loader)
            history.append({'epoch': epoch + 1, 'train_loss': avg_loss})
            pd.DataFrame(history).to_csv(out / f'BERTweet_{stage}_history.csv', index=False)
            print(f'Seed {seed}, {stage}, epoch {epoch + 1}: loss={avg_loss:.4f}', flush=True)
        del optimizer, scheduler, scaler, loader, batch, loss
        gc.collect()
        torch.cuda.empty_cache()
        model.eval()
        return model

    @torch.no_grad()
    def predict_bertweet(model, positions, parameters, description):
        loader = DataLoader(
            EncodedTweets(positions), batch_size=parameters['prediction_batch_size'],
            shuffle=False, num_workers=0
        )
        results = []
        model.eval()
        for batch in tqdm(loader, desc=description):
            batch = {k: v.to(DEVICE) for k, v in batch.items() if k != 'labels'}
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(**batch).logits
            results.append(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
        probability = np.concatenate(results)
        check_probability(probability, positions)
        return probability

    for seed, out, parameters, config, positions, folds in experiments:
        print(f'BERTweet | seed={seed}', flush=True)
        signature = config['signature']
        config_path = out / 'BERTweet_parameters.json'
        if not config_path.exists():
            config_path.write_text(json.dumps(config, indent=2))
        oof = np.full(len(data), np.nan)
        for fold, fit_pos, heldout_pos, path in folds:
            if path.exists():
                probability = read_probabilities(path, signature, heldout_pos)
            else:
                model = fit_bertweet(fit_pos, seed, f'fold_{fold}', out, parameters)
                probability = predict_bertweet(model, heldout_pos, parameters, f'OOF seed {seed}, fold {fold}')
                save_npz(path, signature=np.array(signature), row_positions=heldout_pos, probabilities=probability)
                del model
                gc.collect()
                torch.cuda.empty_cache()
            assert np.isnan(oof[heldout_pos]).all()
            oof[heldout_pos] = probability
        train_pos = positions['train']
        check_probability(oof[train_pos], train_pos)
        oof_path = out / 'PRIVATE_BERTweet_OOF_probabilities.npz'
        if oof_path.exists():
            saved_oof = read_probabilities(oof_path, signature, train_pos)
            assert np.array_equal(saved_oof, oof[train_pos])
            del saved_oof
        else:
            save_npz(oof_path, signature=np.array(signature), row_positions=train_pos, probabilities=oof[train_pos])
        final_path = out / 'PRIVATE_BERTweet_final_probabilities.npz'
        if final_path.exists():
            pass
        else:
            model = fit_bertweet(train_pos, seed, 'final', out, parameters)
            val_probability = predict_bertweet(model, positions['val'], parameters, f'Validation seed {seed}')
            test_probability = predict_bertweet(model, positions['test'], parameters, f'Test seed {seed}')
            save_npz(
                final_path, signature=np.array(signature),
                val_row_positions=positions['val'], val_probabilities=val_probability,
                test_row_positions=positions['test'], test_probabilities=test_probability
            )
            del model, val_probability, test_probability
            gc.collect()
            torch.cuda.empty_cache()
        del oof
        gc.collect()


## 6. Meta-Learner and Threshold

Class-balanced Logistic Regression, C=1. Input: tweet-level OOF probabilities in RF/TextCNN/BERTweet order. Select a threshold from 0.05-0.95 using validation user Macro-F1.


In [15]:
def fit_meta_seed42():
    require_training('meta_42', meta=True)
    import json
    import hashlib
    from pathlib import Path

    import joblib
    import numpy as np
    import pandas as pd
    import sklearn

    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        f1_score, roc_auc_score, confusion_matrix
    )

    BASE = Path("/content/drive_research/MyDrive/thesis/saved_results")
    OUT = BASE / "17_conference_revision/stacking_oof/seed_42"
    SPLITS = BASE / "17_conference_revision/multiseed/seed_42"
    MODEL_ORDER = ["RF", "TextCNN", "BERTweet"]

    data = pd.read_pickle(
        BASE / "02_preprocessing/cleaned_data.pkl"
    )[["username", "label", "cleaned_tweet"]].reset_index(drop=True)

    plan = pd.read_csv(OUT / "PRIVATE_training_user_fold_plan.csv")

    assert data.groupby("username")["label"].nunique().eq(1).all()
    assert plan["username"].is_unique

    def get_positions(part):
        users = pd.read_csv(SPLITS / f"{part}_users.csv")["username"]
        assert users.notna().all() and users.is_unique
        return np.flatnonzero(data["username"].isin(users).to_numpy())

    train_pos = get_positions("train")
    val_pos = get_positions("val")
    test_pos = get_positions("test")

    assert set(data.iloc[train_pos]["username"]) == set(plan["username"])

    for a, b in [(train_pos, val_pos), (train_pos, test_pos), (val_pos, test_pos)]:
        assert set(data.iloc[a]["username"]).isdisjoint(
            set(data.iloc[b]["username"])
        )

    configs = {}
    for name in MODEL_ORDER:
        with open(OUT / f"{name}_parameters.json") as f:
            configs[name] = json.load(f)


    source_bytes = (
        pd.util.hash_pandas_object(data, index=True).values.tobytes()
        + plan.to_csv(index=False).encode()
    )
    expected_rf_signature = hashlib.sha256(
        source_bytes
        + configs["RF"]["sklearn_version"].encode()
        + b"RF100_TFIDF10000_ngram12_minDF2_sublinear_seed42_v1"
    ).hexdigest()

    assert configs["RF"]["signature"] == expected_rf_signature
    for name in ["TextCNN", "BERTweet"]:
        assert configs[name]["source_RF_signature"] == expected_rf_signature

    oof_columns, val_columns = [], []
    diagnostics = []

    for name in MODEL_ORDER:
        with np.load(
            OUT / f"PRIVATE_{name}_OOF_probabilities.npz",
            allow_pickle=False
        ) as saved:
            assert saved["signature"].item() == configs[name]["signature"]
            assert np.array_equal(saved["row_positions"], train_pos)
            oof_probability = saved["probabilities"].copy()

        with np.load(
            OUT / f"PRIVATE_{name}_final_probabilities.npz",
            allow_pickle=False
        ) as saved:
            assert saved["signature"].item() == configs[name]["signature"]
            assert np.array_equal(saved["val_row_positions"], val_pos)
            assert np.array_equal(saved["test_row_positions"], test_pos)
            val_probability = saved["val_probabilities"].copy()

        for values, positions in [
            (oof_probability, train_pos),
            (val_probability, val_pos)
        ]:
            assert values.shape == (len(positions),)
            assert np.isfinite(values).all()
            assert ((values >= 0) & (values <= 1)).all()

        oof_columns.append(oof_probability)
        val_columns.append(val_probability)

        diagnostics.append({
            "model": name,
            "OOF_probability_mean": oof_probability.mean(),
            "OOF_probability_SD": oof_probability.std(),
            "validation_probability_mean": val_probability.mean(),
            "validation_probability_SD": val_probability.std()
        })

    X_oof = np.column_stack(oof_columns)
    X_val = np.column_stack(val_columns)
    y_oof = data.iloc[train_pos]["label"].to_numpy(dtype=int)


    meta_model = LogisticRegression(
        C=1.0,
        solver="lbfgs",
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
    meta_model.fit(X_oof, y_oof)

    positive_column = list(meta_model.classes_).index(1)
    val_probability = meta_model.predict_proba(X_val)[:, positive_column]

    val_frame = data.iloc[val_pos][["username", "label"]].copy()
    val_frame["ASD_probability"] = val_probability

    assert val_frame["username"].nunique() == 37

    def aggregate_validation(threshold):
        frame = val_frame.copy()
        frame["tweet_prediction"] = (
            frame["ASD_probability"] >= threshold
        ).astype(int)

        grouped = frame.groupby("username", sort=True)
        users = grouped.agg(
            label=("label", "first"),
            vote_fraction=("tweet_prediction", "mean"),
            ASD_probability=("ASD_probability", "mean")
        )


        users["prediction"] = (users["vote_fraction"] >= 0.5).astype(int)
        return users

    threshold_rows = []

    for threshold in np.arange(5, 96) / 100:
        users = aggregate_validation(threshold)
        threshold_rows.append({
            "threshold": threshold,
            "user_macro_f1": f1_score(
                users["label"], users["prediction"],
                average="macro", labels=[0, 1], zero_division=0
            )
        })

    threshold_table = pd.DataFrame(threshold_rows)
    maximum_score = threshold_table["user_macro_f1"].max()

    candidates = threshold_table.loc[
        np.isclose(
            threshold_table["user_macro_f1"],
            maximum_score,
            rtol=0, atol=1e-12
        )
    ].copy()


    candidates["distance_from_050"] = (
        candidates["threshold"] - 0.5
    ).abs().round(10)

    best_threshold = float(
        candidates.sort_values(
            ["distance_from_050", "threshold"]
        ).iloc[0]["threshold"]
    )

    selected_users = aggregate_validation(best_threshold)
    default_users = aggregate_validation(0.50)

    coefficients = pd.DataFrame({
        "base_model": MODEL_ORDER,
        "coefficient": meta_model.coef_[0]
    })

    threshold_table.to_csv(
        OUT / "stacking_validation_threshold_search.csv", index=False
    )
    coefficients.to_csv(
        OUT / "meta_learner_coefficients.csv", index=False
    )
    pd.DataFrame(diagnostics).to_csv(
        OUT / "base_probability_diagnostics.csv", index=False
    )
    selected_users.to_csv(
        OUT / "PRIVATE_stacking_validation_user_predictions.csv"
    )
    joblib.dump(meta_model, OUT / "PRIVATE_meta_model.joblib")

    settings = {
        "seed": 42,
        "model_order": MODEL_ORDER,
        "meta_parameters": meta_model.get_params(),
        "meta_intercept": float(meta_model.intercept_[0]),
        "sklearn_version": sklearn.__version__,
        "base_signatures": {
            name: configs[name]["signature"] for name in MODEL_ORDER
        },
        "selected_threshold": best_threshold,
        "validation_user_macro_f1": float(maximum_score),
        "threshold_selection_metric": "user-level Macro-F1",
        "threshold_grid": {"start": 0.05, "stop": 0.95, "step": 0.01},
        "threshold_tie_rule": "closest to 0.50, then smaller",
        "user_rule": "majority vote; ties assigned ASD",
        "test_metrics_evaluated": False
    }

    with open(OUT / "stacking_selected_settings.json", "w") as f:
        json.dump(settings, f, indent=2)

    print('Base probabilities')
    print(pd.DataFrame(diagnostics).round(4).to_string(index=False))

    print('Coefficients')
    print(coefficients.round(4).to_string(index=False))
    print('Intercept:', round(float(meta_model.intercept_[0]), 4))

    print('Validation | users=37')
    print('Threshold:', best_threshold)
    print('User Macro-F1:', round(maximum_score, 4))
    print('User Macro-F1 threshold 0.50:', round(f1_score(default_users['label'], default_users['prediction'], average='macro', labels=[0, 1], zero_division=0), 4))
    print('User ROC-AUC:', round(roc_auc_score(selected_users['label'], selected_users['ASD_probability']), 4))
    print('Validation confusion matrix [non-ASD, ASD]')
    print(confusion_matrix(selected_users['label'], selected_users['prediction'], labels=[0, 1]))


In [16]:
def fit_meta_seed43_46():
    require_training('meta_43_46', meta=True)
    import gc
    import json
    import hashlib
    import tempfile
    import os
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import sklearn
    import joblib
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix

    BASE = Path('/content/drive_research/MyDrive/thesis/saved_results')
    if not (BASE / '02_preprocessing/cleaned_data.pkl').is_file():
        from google.colab import drive
        drive.mount('/content/drive_research')
    ROOT = BASE / '17_conference_revision'
    MODEL_ORDER = ['RF', 'TextCNN', 'BERTweet']
    data = pd.read_pickle(BASE / '02_preprocessing/cleaned_data.pkl')[
        ['username', 'label', 'cleaned_tweet']
    ].reset_index(drop=True)
    assert data.notna().all().all()
    assert data['label'].isin([0, 1]).all()
    assert data.groupby('username')['label'].nunique().eq(1).all()
    assert data['username'].nunique() == 244
    source_data = pd.util.hash_pandas_object(data, index=True).values.tobytes()


    GRID = np.arange(5, 96) / 100

    def choose_threshold(table):
        maximum = table['user_macro_f1'].max()
        candidates = table.loc[np.isclose(
            table['user_macro_f1'], maximum, rtol=0, atol=1e-12
        )].copy()
        candidates['distance'] = (candidates['threshold'] - 0.5).abs().round(10)
        best = candidates.sort_values(['distance', 'threshold']).iloc[0]
        return float(best['threshold']), float(maximum)

    def read_probability(path, signature, positions, prefix=None):
        with np.load(path, allow_pickle=False) as saved:
            assert saved['signature'].item() == signature, path.name
            row_key = 'row_positions' if prefix is None else f'{prefix}_row_positions'
            probability_key = 'probabilities' if prefix is None else f'{prefix}_probabilities'
            assert np.array_equal(saved[row_key], positions), path.name
            values = saved[probability_key].copy()
        assert values.shape == (len(positions),)
        assert np.isfinite(values).all()
        assert ((values >= 0) & (values <= 1)).all()
        return values

    def save_table(path, frame, index=False):

        if path.exists():
            saved = pd.read_csv(path, index_col=0 if index else None)
            pd.testing.assert_frame_equal(
                saved, frame, check_dtype=False, check_exact=False,
                rtol=1e-9, atol=1e-10
            )
            return
        temporary = None
        try:
            with tempfile.NamedTemporaryFile(
                mode='w', dir=path.parent, suffix='.tmp', delete=False
            ) as handle:
                temporary = Path(handle.name)
                frame.to_csv(handle, index=index)
            os.replace(temporary, path)
        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)

    runs = []
    for seed in range(43, 47):
        out = ROOT / f'stacking_oof/seed_{seed}'
        split_dir = ROOT / f'multiseed/seed_{seed}'
        plan = pd.read_csv(out / 'PRIVATE_training_user_fold_plan.csv')
        assert plan['username'].is_unique
        assert plan['heldout_fold'].isin([1, 2, 3]).all()
        parts = {}
        for part in ['train', 'val', 'test']:
            table = pd.read_csv(split_dir / f'{part}_users.csv')
            assert table['username'].notna().all() and table['username'].is_unique
            parts[part] = set(table['username'])
        assert not (parts['train'] & parts['val'])
        assert not (parts['train'] & parts['test'])
        assert not (parts['val'] & parts['test'])
        assert set().union(*parts.values()) == set(data['username'])
        assert tuple(len(parts[p]) for p in ['train', 'val', 'test']) == (146, 37, 61)
        assert set(plan['username']) == parts['train']
        labels_by_user = data.groupby('username')['label'].first()
        assert np.array_equal(plan['label'], labels_by_user.loc[plan['username']])
        positions = {
            part: np.flatnonzero(data['username'].isin(names).to_numpy())
            for part, names in parts.items()
        }
        source_bytes = (
            source_data + plan.to_csv(index=False).encode()
            + json.dumps({p: sorted(v) for p, v in parts.items()}, sort_keys=True).encode()
        )
        configs = {
            name: json.loads((out / f'{name}_parameters.json').read_text())
            for name in MODEL_ORDER
        }
        assert configs['RF']['sklearn_version'] == sklearn.__version__
        expected_rf = hashlib.sha256(
            source_bytes + json.dumps(configs['RF']['parameters'], sort_keys=True).encode()
        ).hexdigest()
        assert configs['RF']['signature'] == expected_rf
        for name in ['TextCNN', 'BERTweet']:
            assert configs[name]['source_RF_signature'] == expected_rf
        expected_cnn = hashlib.sha256(
            source_bytes + expected_rf.encode()
            + json.dumps(configs['TextCNN']['parameters'], sort_keys=True).encode()
            + json.dumps(configs['TextCNN']['tokenizer_settings'], sort_keys=True).encode()
        ).hexdigest()
        assert configs['TextCNN']['signature'] == expected_cnn
        bert_unsigned = {k: v for k, v in configs['BERTweet'].items() if k != 'signature'}
        expected_bert = hashlib.sha256(
            source_bytes + json.dumps(bert_unsigned, sort_keys=True).encode()
        ).hexdigest()
        assert configs['BERTweet']['signature'] == expected_bert
        oof_cols, val_cols, diagnostic = [], [], []
        for name in MODEL_ORDER:
            signature = configs[name]['signature']
            oof = read_probability(
                out / f'PRIVATE_{name}_OOF_probabilities.npz', signature, positions['train']
            )
            val = read_probability(
                out / f'PRIVATE_{name}_final_probabilities.npz', signature, positions['val'], 'val'
            )

            with np.load(out / f'PRIVATE_{name}_final_probabilities.npz', allow_pickle=False) as saved:
                assert np.array_equal(saved['test_row_positions'], positions['test'])
            for fold in [1, 2, 3]:
                heldout = set(plan.loc[plan['heldout_fold'].eq(fold), 'username'])
                heldout_pos = np.flatnonzero(data['username'].isin(heldout).to_numpy())
                fold_values = read_probability(
                    out / f'PRIVATE_{name}_fold_{fold}_probabilities.npz', signature, heldout_pos
                )
                assert np.array_equal(oof[np.searchsorted(positions['train'], heldout_pos)], fold_values)
            oof_cols.append(oof)
            val_cols.append(val)
            diagnostic.append({
                'model': name, 'OOF_probability_mean': float(oof.mean()),
                'OOF_probability_SD': float(oof.std()),
                'validation_probability_mean': float(val.mean()),
                'validation_probability_SD': float(val.std())
            })
        X_oof, X_val = np.column_stack(oof_cols), np.column_stack(val_cols)
        y_oof = data.iloc[positions['train']]['label'].to_numpy(dtype=int)
        meta_parameters = {
            'C': 1.0, 'solver': 'lbfgs', 'class_weight': 'balanced',
            'max_iter': 1000, 'random_state': seed
        }
        input_signature = hashlib.sha256(
            source_bytes + X_oof.tobytes() + X_val.tobytes()
            + json.dumps(meta_parameters, sort_keys=True).encode()
            + sklearn.__version__.encode()
        ).hexdigest()
        settings_path = out / 'stacking_selected_settings.json'
        previous = json.loads(settings_path.read_text()) if settings_path.exists() else None
        if previous is not None:
            assert previous.get('input_signature') == input_signature, 'Meta input or configuration changed.'
            assert previous['base_signatures'] == {n: configs[n]['signature'] for n in MODEL_ORDER}
            assert previous.get('test_metrics_evaluated') is False, (
                'Test evaluation already recorded; threshold selection is locked.'
            )
        runs.append((seed, out, positions, configs, X_oof, X_val, y_oof,
                     meta_parameters, input_signature, diagnostic, previous))

    summary_rows = []
    for (seed, out, positions, configs, X_oof, X_val, y_oof,
         meta_parameters, input_signature, diagnostic, previous) in runs:
        model_path = out / 'PRIVATE_meta_model.joblib'
        if model_path.exists():
            meta = joblib.load(model_path)
            for key, value in meta_parameters.items():
                assert meta.get_params()[key] == value

            assert getattr(meta, 'research_input_signature_', None) == input_signature
        else:
            assert previous is None, 'Missing meta checkpoint.'
            meta = LogisticRegression(**meta_parameters)
            meta.fit(X_oof, y_oof)
            meta.research_input_signature_ = input_signature

            with tempfile.NamedTemporaryFile(dir=out, suffix='.tmp', delete=False) as h:
                temporary = Path(h.name)
            try:
                joblib.dump(meta, temporary)
                os.replace(temporary, model_path)
            finally:
                temporary.unlink(missing_ok=True)
        assert list(meta.classes_) == [0, 1]
        val_probability = meta.predict_proba(X_val)[:, 1]
        assert np.isfinite(val_probability).all()
        val_frame = data.iloc[positions['val']][['username', 'label']].copy()
        val_frame['ASD_probability'] = val_probability

        def aggregate_validation(threshold):
            frame = val_frame.copy()
            frame['tweet_prediction'] = (frame['ASD_probability'] >= threshold).astype(int)
            users = frame.groupby('username', sort=True).agg(
                label=('label', 'first'), vote_fraction=('tweet_prediction', 'mean'),
                ASD_probability=('ASD_probability', 'mean')
            )
            users['prediction'] = (users['vote_fraction'] >= 0.5).astype(int)
            assert len(users) == 37
            return users

        threshold_rows = []
        for threshold in GRID:
            users = aggregate_validation(threshold)
            threshold_rows.append({
                'threshold': float(threshold),
                'user_macro_f1': float(f1_score(
                    users['label'], users['prediction'], average='macro',
                    labels=[0, 1], zero_division=0
                ))
            })
        threshold_table = pd.DataFrame(threshold_rows)
        selected_threshold, maximum = choose_threshold(threshold_table)
        users = aggregate_validation(selected_threshold)
        default_users = aggregate_validation(0.50)
        default_score = float(f1_score(
            default_users['label'], default_users['prediction'],
            average='macro', labels=[0, 1], zero_division=0
        ))
        val_auc = float(roc_auc_score(users['label'], users['ASD_probability']))
        coefficients = pd.DataFrame({'base_model': MODEL_ORDER, 'coefficient': meta.coef_[0]})
        settings = {
            'seed': seed, 'input_signature': input_signature,
            'model_order': MODEL_ORDER, 'meta_parameters': meta.get_params(),
            'meta_intercept': float(meta.intercept_[0]),
            'sklearn_version': sklearn.__version__,
            'base_signatures': {n: configs[n]['signature'] for n in MODEL_ORDER},
            'selected_threshold': selected_threshold,
            'validation_user_macro_f1': maximum,
            'threshold_selection_metric': 'user-level Macro-F1',
            'threshold_grid': {'start': 0.05, 'stop': 0.95, 'step': 0.01},
            'threshold_tie_rule': 'closest to 0.50, then smaller',
            'user_rule': 'majority vote; ties assigned ASD',
            'test_metrics_evaluated': False
        }
        if previous is not None:
            assert previous == settings, 'Settings mismatch.'
        save_table(out / 'stacking_validation_threshold_search.csv', threshold_table)
        save_table(out / 'meta_learner_coefficients.csv', coefficients)
        save_table(out / 'base_probability_diagnostics.csv', pd.DataFrame(diagnostic))
        save_table(out / 'PRIVATE_stacking_validation_user_predictions.csv', users, index=True)
        path = out / 'stacking_selected_settings.json'
        if not path.exists():
            with tempfile.NamedTemporaryFile(mode='w', dir=out, suffix='.tmp', delete=False) as h:
                temporary = Path(h.name)
                json.dump(settings, h, indent=2)
            os.replace(temporary, path)
        print(f'Seed {seed} | validation', flush=True)
        print('Threshold:', selected_threshold)
        print('User Macro-F1:', round(maximum, 4))
        print('User Macro-F1 at threshold 0.50:', round(default_score, 4))
        print('User ROC-AUC:', round(val_auc, 4))
        print('Coefficients')
        print(coefficients.round(4).to_string(index=False))
        print('Intercept:', round(float(meta.intercept_[0]), 4))
        print('Confusion matrix [non-ASD, ASD]:')
        print(confusion_matrix(users['label'], users['prediction'], labels=[0, 1]))
        summary_rows.append({
            'seed': seed, 'threshold': selected_threshold,
            'validation_user_macro_f1': maximum,
            'validation_user_macro_f1_at_050': default_score,
            'validation_user_roc_auc': val_auc
        })
        gc.collect()
    summary = pd.DataFrame(summary_rows)
    save_table(ROOT / 'stacking_oof/validation_summary_seed43_46.csv', summary)
    print('Validation summary')
    print(summary.round(4).to_string(index=False))


In [17]:
TRAINING_STAGES = {
    'folds_42':prepare_folds_seed42,'folds_43_46':prepare_folds_seed43_46,
    'rf_42':train_rf_seed42,'rf_43_46':train_rf_seed43_46,
    'textcnn_42':train_textcnn_seed42,'textcnn_43_46':train_textcnn_seed43_46,
    'bertweet_42':train_bertweet_seed42,'bertweet_43_46':train_bertweet_seed43_46,
    'meta_42':fit_meta_seed42,'meta_43_46':fit_meta_seed43_46
}
if ENABLE_TRAINING:
    if TRAINING_STAGE not in TRAINING_STAGES:
        raise ValueError('Select a valid TRAINING_STAGE.')
    TRAINING_STAGES[TRAINING_STAGE]()


### Meta-Learner Results

Model order, coefficients, validation Macro-F1, and selected thresholds.


In [18]:
show_result('Meta-Learner Settings and Coefficients',saved_meta_overview)
show_result('Base Probability Diagnostics',saved_probability_diagnostics)


Meta-Learner Settings and Coefficients


,seed,threshold,validation_user_macro_f1,intercept
0,42,0.49,0.7633,-0.6051
1,43,0.46,0.7688,-1.0229
2,44,0.46,0.6022,-1.2405
3,45,0.50,0.5800,-1.1010
4,46,0.45,0.8103,-1.1598


base_model,RF,TextCNN,BERTweet
seed,,,
42,0.5988,0.1301,0.2780
43,0.3055,0.1933,1.1743
44,0.3455,0.1087,1.6828
45,0.3439,0.0602,1.4852
46,0.3867,0.0114,1.6049


Base Probability Diagnostics


,seed,model,OOF_probability_mean,OOF_probability_SD,validation_probability_mean,validation_probability_SD
0,42,RF,0.5928,0.1658,0.5900,0.1554
1,42,TextCNN,0.6485,0.3930,0.6208,0.3942
2,42,BERTweet,0.6386,0.2593,0.6902,0.2886
3,43,RF,0.5921,0.1696,0.5913,0.1629
4,43,TextCNN,0.6357,0.3984,0.6282,0.3851
5,43,BERTweet,0.6704,0.3032,0.6758,0.2959
6,44,RF,0.5657,0.1749,0.5605,0.1708
7,44,TextCNN,0.6101,0.4031,0.6480,0.3869
8,44,BERTweet,0.6590,0.3119,0.6520,0.3151
9,45,RF,0.5758,0.1647,0.5578,0.1609


,seed,model,OOF_probability_mean,OOF_probability_SD,validation_probability_mean,validation_probability_SD
0,42,RF,0.5928,0.1658,0.5900,0.1554
1,42,TextCNN,0.6485,0.3930,0.6208,0.3942
2,42,BERTweet,0.6386,0.2593,0.6902,0.2886
3,43,RF,0.5921,0.1696,0.5913,0.1629
4,43,TextCNN,0.6357,0.3984,0.6282,0.3851
5,43,BERTweet,0.6704,0.3032,0.6758,0.2959
6,44,RF,0.5657,0.1749,0.5605,0.1708
7,44,TextCNN,0.6101,0.4031,0.6480,0.3869
8,44,BERTweet,0.6590,0.3119,0.6520,0.3151
9,45,RF,0.5758,0.1647,0.5578,0.1609


## 7. Prediction

Base-model probabilities and validation-selected thresholds produce tweet and user predictions.


In [19]:
def checkpoint_textcnn(out, stage, parameters, positions, expected=None):
    model_path = out/f'PRIVATE_TextCNN_{stage}.keras'
    tokenizer_path = out/f'PRIVATE_TextCNN_{stage}_tokenizer.json'
    require_files([model_path,tokenizer_path])
    import tensorflow as tf
    from tensorflow.keras.preprocessing.text import tokenizer_from_json
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    for key,value in {'tensorflow_version':tf.__version__,'keras_version':tf.keras.__version__,
                      'numpy_version':np.__version__,'pandas_version':pd.__version__}.items():
        assert parameters[key]==value, (key,parameters[key],value)
    for gpu in tf.config.list_physical_devices('GPU'):
        try:
            tf.config.experimental.set_memory_growth(gpu,True)
        except RuntimeError:
            pass
    tf.keras.mixed_precision.set_global_policy('float32')
    tokenizer = tokenizer_from_json(tokenizer_path.read_text())
    assert tokenizer.num_words==parameters['vocabulary_limit'] and tokenizer.oov_token=='<OOV>'
    model = tf.keras.models.load_model(model_path,compile=False)
    sequences = tokenizer.texts_to_sequences(data.iloc[positions]['cleaned_tweet'].tolist())
    X = pad_sequences(sequences,maxlen=parameters['sequence_length'],padding=parameters['padding'],truncating=parameters['truncating'])
    probability = model.predict(X,batch_size=parameters['prediction_batch_size'],verbose=0).reshape(-1)
    assert probability.shape==(len(positions),) and np.isfinite(probability).all()
    assert ((probability>=0)&(probability<=1)).all()
    del X,sequences,model,tokenizer
    tf.keras.backend.clear_session()
    gc.collect()
    if expected is not None:
        # Float32 verification only; evaluation keeps the saved probabilities.
        np.testing.assert_allclose(probability,expected,rtol=1e-5,atol=1e-6,
                                   err_msg=f'Checkpoint mismatch ({out.name}/{stage}). Use PREDICT_FROM_CHECKPOINT=False to evaluate saved probabilities.')
        return expected
    return probability

def base_probability(out,name,stage,signature,positions,parameters=None,prefix=None):
    path = out/(f'PRIVATE_{name}_final_probabilities.npz' if stage=='final'
                else f'PRIVATE_{name}_{stage}_probabilities.npz')
    expected = read_probability(path,signature,positions,prefix) if path.is_file() else None
    complete = name=='TextCNN' and all(p.is_file() for p in [out/f'PRIVATE_TextCNN_{stage}.keras',
                                                          out/f'PRIVATE_TextCNN_{stage}_tokenizer.json'])
    if complete and (PREDICT_FROM_CHECKPOINT or expected is None):
        return checkpoint_textcnn(out,stage,parameters,positions,expected)
    if expected is None:
        needed=[path]
        if name=='TextCNN':
            needed.extend([out/f'PRIVATE_TextCNN_{stage}.keras',out/f'PRIVATE_TextCNN_{stage}_tokenizer.json'])
        require_files(needed)
    return expected


In [20]:
def aggregate(frame, probability, threshold):
    temp = frame[['username', 'label']].copy()
    temp['ASD_probability'] = probability
    temp['tweet_prediction'] = (probability >= threshold).astype(int)
    users = temp.groupby('username', sort=True).agg(
        label=('label', 'first'), vote_fraction=('tweet_prediction', 'mean'),
        ASD_probability=('ASD_probability', 'mean'), n_tweets=('label', 'size')
    )
    users['prediction'] = (users['vote_fraction'] >= 0.5).astype(int)
    return users


In [21]:
def predict_stacking(data,seed):
    out,positions,plan,configs,source_bytes = stacking_context(data,seed)
    settings = frozen_settings(seed)
    assert settings['base_signatures']=={n:configs[n]['signature'] for n in MODEL_ORDER}
    require_files([out/'PRIVATE_meta_model.joblib',out/'meta_learner_coefficients.csv'])
    assert settings['sklearn_version']==sklearn.__version__, 'scikit-learn version does not match the checkpoint.'
    meta = joblib.load(out/'PRIVATE_meta_model.joblib')
    assert meta.get_params()==settings['meta_parameters'] and list(meta.classes_)==[0,1]
    coefficients = pd.read_csv(out/'meta_learner_coefficients.csv')
    assert coefficients['base_model'].tolist()==MODEL_ORDER
    assert np.allclose(meta.coef_[0],coefficients['coefficient'],rtol=0,atol=1e-10)
    assert np.isclose(meta.intercept_[0],settings['meta_intercept'],rtol=0,atol=1e-12)
    columns = {'train':[],'val':[],'test':[]}
    for name in MODEL_ORDER:
        signature = configs[name]['signature']
        oof_path = out/f'PRIVATE_{name}_OOF_probabilities.npz'
        if oof_path.is_file():
            oof = read_probability(oof_path,signature,positions['train'])
        elif name=='TextCNN':
            complete_oof = np.full(len(data),np.nan)
            for fold in [1,2,3]:
                heldout = set(plan.loc[plan['heldout_fold'].eq(fold),'username'])
                heldout_pos = np.flatnonzero(data['username'].isin(heldout).to_numpy())
                complete_oof[heldout_pos] = base_probability(out,name,f'fold_{fold}',signature,
                                                           heldout_pos,configs[name]['parameters'])
            oof = complete_oof[positions['train']]
        else:
            require_files([oof_path])
        for fold in [1,2,3]:
            path = out/f'PRIVATE_{name}_fold_{fold}_probabilities.npz'
            if path.is_file():
                heldout = set(plan.loc[plan['heldout_fold'].eq(fold),'username'])
                heldout_pos = np.flatnonzero(data['username'].isin(heldout).to_numpy())
                values = read_probability(path,signature,heldout_pos)
                assert np.array_equal(oof[np.searchsorted(positions['train'],heldout_pos)],values)
        columns['train'].append(oof)
        for part in ['val','test']:
            columns[part].append(base_probability(out,name,'final',signature,positions[part],
                                                 configs[name].get('parameters'),prefix=part))
    X_oof,X_val,X_test = [np.column_stack(columns[p]) for p in ['train','val','test']]
    if seed!=42:
        params={'C':1.0,'solver':'lbfgs','class_weight':'balanced','max_iter':1000,'random_state':seed}
        input_signature = hashlib.sha256(source_bytes+X_oof.tobytes()+X_val.tobytes()
            +json.dumps(params,sort_keys=True).encode()+settings['sklearn_version'].encode()).hexdigest()
        assert input_signature==settings['input_signature']
        assert getattr(meta,'research_input_signature_',None)==input_signature
    probability = meta.predict_proba(X_test)[:,list(meta.classes_).index(1)]
    threshold = float(settings['selected_threshold'])
    frame = data.iloc[positions['test']][['username','label']].copy()
    users = aggregate(frame,probability,threshold)
    if seed==42:
        users = users.drop(columns=['n_tweets'])
    users = check_users(users,data,positions['test'])
    return users, probability, frame

def load_stacking_users(data,seed):
    out = STACKING/f'seed_{seed}'
    settings = frozen_settings(seed)
    saved_path = out/'PRIVATE_stacking_test_user_predictions.csv'
    core = [out/'PRIVATE_training_user_fold_plan.csv',out/'PRIVATE_meta_model.joblib',out/'meta_learner_coefficients.csv']
    core += [out/f'{name}_parameters.json' for name in MODEL_ORDER]
    for name in ['RF','BERTweet']:
        core += [out/f'PRIVATE_{name}_OOF_probabilities.npz',out/f'PRIVATE_{name}_final_probabilities.npz']
    cnn_oof = (out/'PRIVATE_TextCNN_OOF_probabilities.npz').is_file() or all(
        (out/f'PRIVATE_TextCNN_fold_{f}.keras').is_file() and
        (out/f'PRIVATE_TextCNN_fold_{f}_tokenizer.json').is_file() for f in [1,2,3])
    cnn_final = (out/'PRIVATE_TextCNN_final_probabilities.npz').is_file() or all(
        p.is_file() for p in [out/'PRIVATE_TextCNN_final.keras',out/'PRIVATE_TextCNN_final_tokenizer.json'])
    if all(p.is_file() for p in core) and cnn_oof and cnn_final:
        users,probability,frame = predict_stacking(data,seed)
        if saved_path.is_file():
            saved = pd.read_csv(saved_path,index_col=0)
            saved = check_users(saved,data,partitions[seed][1]['test'])
            pd.testing.assert_frame_equal(saved,users,check_dtype=False,rtol=1e-9,atol=1e-10)
        return users,probability,frame,'meta checkpoint + probabilities'
    if saved_path.is_file():
        users = pd.read_csv(saved_path,index_col=0)
        users = check_users(users,data,partitions[seed][1]['test'])
        return users,None,None,'saved user predictions (base artifacts unavailable)'
    needed = core+[saved_path,out/'PRIVATE_TextCNN_OOF_probabilities.npz',out/'PRIVATE_TextCNN_final_probabilities.npz']
    require_files(needed)
    raise FileNotFoundError('TextCNN probabilities or checkpoint required.')


## 8. Evaluation and Bootstrap

Majority vote; ties are assigned to ASD. AUC uses mean probabilities. CIs use 5,000 stratified user bootstrap resamples. Set RECOMPUTE_CI=True to recompute CIs.


In [22]:
def calculate_metrics(y, pred, prob):
    tp = np.sum((y == 1) & (pred == 1))
    tn = np.sum((y == 0) & (pred == 0))
    fp = np.sum((y == 0) & (pred == 1))
    fn = np.sum((y == 1) & (pred == 0))

    def divide(a, b):
        return float(a / b) if b else 0.0

    f1_asd = divide(2 * tp, 2 * tp + fp + fn)
    f1_non = divide(2 * tn, 2 * tn + fp + fn)

    return {
        "accuracy": divide(tp + tn, len(y)),
        "precision_ASD": divide(tp, tp + fp),
        "recall_ASD": divide(tp, tp + fn),
        "f1_ASD": f1_asd,
        "macro_f1": (f1_asd + f1_non) / 2,
        "recall_non_ASD": divide(tn, tn + fp),
        "f1_non_ASD": f1_non,
        "roc_auc": roc_auc_score(y, prob)
    }

def scores(y, prediction, probability):
    return {
        'accuracy': accuracy_score(y, prediction),
        'precision_ASD': precision_score(y, prediction, zero_division=0),
        'recall_ASD': recall_score(y, prediction, zero_division=0),
        'f1_ASD': f1_score(y, prediction, zero_division=0),
        'macro_f1': f1_score(y, prediction, average='macro', labels=[0, 1], zero_division=0),
        'recall_non_ASD': recall_score(y, prediction, pos_label=0, zero_division=0),
        'f1_non_ASD': f1_score(y, prediction, pos_label=0, zero_division=0),
        'roc_auc': roc_auc_score(y, probability)
    }


In [23]:
def bootstrap_samples(y, prediction, probability, n_bootstrap, seed):

    y, prediction, probability = map(np.asarray, [y, prediction, probability])
    negative, positive = np.flatnonzero(y == 0), np.flatnonzero(y == 1)
    assert len(negative) > 0 and len(positive) > 0
    rng = np.random.default_rng(seed)
    neg_idx = rng.choice(negative, size=(n_bootstrap, len(negative)), replace=True)
    pos_idx = rng.choice(positive, size=(n_bootstrap, len(positive)), replace=True)
    tp = prediction[pos_idx].sum(axis=1).astype(float)
    fn = len(positive) - tp
    fp = prediction[neg_idx].sum(axis=1).astype(float)
    tn = len(negative) - fp
    def divide(a, b):
        return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b != 0)
    f1_positive = divide(2 * tp, 2 * tp + fp + fn)
    f1_negative = divide(2 * tn, 2 * tn + fp + fn)

    pos_prob = probability[pos_idx][:, :, None]
    neg_prob = probability[neg_idx][:, None, :]
    auc = ((pos_prob > neg_prob).sum(axis=(1, 2))
           + 0.5 * (pos_prob == neg_prob).sum(axis=(1, 2))) / (len(positive) * len(negative))
    return {
        'accuracy': (tp + tn) / len(y),
        'precision_ASD': divide(tp, tp + fp),
        'recall_ASD': tp / len(positive),
        'f1_ASD': f1_positive,
        'macro_f1': (f1_positive + f1_negative) / 2,
        'recall_non_ASD': tn / len(negative),
        'f1_non_ASD': f1_negative,
        'roc_auc': auc
    }


In [24]:
def bootstrap_seed42(users):
    y = users['label'].to_numpy()
    pred = users['prediction'].to_numpy()
    prob = users['ASD_probability'].to_numpy()
    user_metrics = calculate_metrics(y,pred,prob)
    rng = np.random.default_rng(42)
    asd_idx = np.flatnonzero(y==1)
    non_idx = np.flatnonzero(y==0)
    bootstrap_rows = []
    for _ in range(5000):
        sampled = np.concatenate([
            rng.choice(asd_idx,len(asd_idx),replace=True),
            rng.choice(non_idx,len(non_idx),replace=True)
        ])
        bootstrap_rows.append(calculate_metrics(y[sampled],pred[sampled],prob[sampled]))
    bootstrap = pd.DataFrame(bootstrap_rows)
    ci_rows = []
    for metric,estimate in user_metrics.items():
        low,high = np.percentile(bootstrap[metric],[2.5,97.5])
        ci_rows.append({'metric':metric,'estimate':estimate,'CI_low':low,'CI_high':high,
                        'n_bootstrap':5000,'bootstrap_seed':42})
    return pd.DataFrame(ci_rows)

def stacking_ci(users,seed,values):
    path = STACKING/f'seed_{seed}/stacking_test_user_bootstrap_CI.csv'
    if path.is_file() and not RECOMPUTE_CI:
        ci = pd.read_csv(path)
        assert ci['metric'].is_unique and set(ci['metric'])==set(METRICS)
        for _,row in ci.iterrows():
            assert np.isclose(row['estimate'],values[row['metric']],rtol=0,atol=1e-8)
            assert 0<=row['CI_low']<=row['CI_high']<=1
        if seed==42:
            assert ci['n_bootstrap'].eq(5000).all() and ci['bootstrap_seed'].eq(42).all()
        else:
            assert ci['seed'].eq(seed).all() and ci['model'].eq('OOF Stacking').all()
        return ci
    if seed==42:
        ci = bootstrap_seed42(users)
    else:
        boot = bootstrap_samples(users['label'],users['prediction'],users['ASD_probability'],5000,seed)
        rows=[]
        for metric in METRICS:
            low,high = np.percentile(boot[metric],[2.5,97.5])
            rows.append({'model':'OOF Stacking','seed':seed,'metric':metric,'estimate':values[metric],
                         'CI_low':float(low),'CI_high':float(high)})
        ci = pd.DataFrame(rows)
    if path.is_file():
        stored = pd.read_csv(path).set_index('metric').loc[METRICS]
        actual = ci.set_index('metric').loc[METRICS]
        np.testing.assert_allclose(stored[['estimate','CI_low','CI_high']],actual[['estimate','CI_low','CI_high']],rtol=1e-9,atol=1e-10)
    return ci


In [25]:
rows,ci_tables,status_rows = [],[],[]
for seed in SEEDS:
    users,probability,frame,basis = load_stacking_users(data,seed)
    settings = frozen_settings(seed)
    threshold = float(settings['selected_threshold'])
    values = calculate_metrics(users['label'].to_numpy(),users['prediction'].to_numpy(),
                               users['ASD_probability'].to_numpy()) if seed==42 else scores(
        users['label'],users['prediction'],users['ASD_probability'])
    path = STACKING/f'seed_{seed}/stacking_test_metrics.csv'
    if path.is_file():
        stored = pd.read_csv(path)
        user_row = stored.loc[stored['level'].astype(str).str.lower().str.replace(r'[^a-z]','',regex=True).isin(['user','userlevel'])]
        assert len(user_row)==1
        check_metric_row(user_row.iloc[0],values)
        if probability is not None:
            tweet_values = calculate_metrics(frame['label'].to_numpy(),(probability>=threshold).astype(int),probability) if seed==42 else scores(
                frame['label'],(probability>=threshold).astype(int),probability)
            tweet_row = stored.loc[stored['level'].eq('tweet')]
            assert len(tweet_row)==1
            check_metric_row(tweet_row.iloc[0],tweet_values)
    if probability is not None:
        tweet_values = calculate_metrics(frame['label'].to_numpy(),(probability>=threshold).astype(int),probability) if seed==42 else scores(
            frame['label'],(probability>=threshold).astype(int),probability)
        print(f'Seed {seed} - Tweet Metrics')
        display(pd.DataFrame([tweet_values]).round(4))
    elif path.is_file():
        print(f'Seed {seed} - Tweet Metrics')
        display(stored.loc[stored['level'].eq('tweet')].round(4))
    else:
        print(f'Seed {seed}: tweet metrics require stacking_test_metrics.csv or tweet probabilities.')
    ci = stacking_ci(users,seed,values)
    matrix = confusion_matrix(users['label'],users['prediction'],labels=[0,1])
    cm_path = STACKING/f'seed_{seed}/stacking_test_user_confusion_matrix.csv'
    if cm_path.is_file():
        assert np.array_equal(pd.read_csv(cm_path,index_col=0).to_numpy(),matrix)
    print(f'Seed {seed} | threshold {threshold:.2f}')
    display(ci[['metric','estimate','CI_low','CI_high']].round(4))
    display(pd.DataFrame(matrix,index=['actual_non_ASD','actual_ASD'],columns=['predicted_non_ASD','predicted_ASD']))
    rows.append({'model':'OOF Stacking','seed':seed,'threshold':threshold,**values})
    ci_tables.append(ci)
    cnn_checkpoint = all((STACKING/f'seed_{seed}'/f).is_file() for f in ['PRIVATE_TextCNN_final.keras','PRIVATE_TextCNN_final_tokenizer.json'])
    status_rows.append({'seed':seed,'basis':basis,
                        'TextCNN': 'checkpoint + tokenizer' if probability is not None and PREDICT_FROM_CHECKPOINT and cnn_checkpoint else 'saved probabilities/predictions',
                        'CI':'computed' if RECOMPUTE_CI or not (STACKING/f'seed_{seed}/stacking_test_user_bootstrap_CI.csv').is_file() else 'saved'})
    del users,probability,frame
    gc.collect()
results = pd.DataFrame(rows).sort_values('seed').reset_index(drop=True)
summary = pd.DataFrame([{'metric':m,'mean':results[m].mean(),'std':results[m].std(ddof=1)} for m in METRICS])
assert results['seed'].tolist()==[42,43,44,45,46]
for filename,table in [('stacking_multiseed_results.csv',results),('stacking_multiseed_mean_SD.csv',summary)]:
    path = STACKING/filename
    if path.is_file():
        stored = pd.read_csv(path)
        if 'metric' in stored:
            pd.testing.assert_frame_equal(stored,table,check_dtype=False,rtol=1e-9,atol=1e-10)
        else:
            pd.testing.assert_frame_equal(stored[table.columns],table,check_dtype=False,rtol=1e-9,atol=1e-10)
display(results[['seed','threshold']+METRICS].round(4))
display(summary.round(4))
display(pd.DataFrame(status_rows))


Seed 42 - Tweet Metrics


,accuracy,precision_ASD,recall_ASD,f1_ASD,macro_f1,recall_non_ASD,f1_non_ASD,roc_auc
0,0.649,0.9434,0.6592,0.7761,0.4822,0.5274,0.1883,0.641


Seed 42 | threshold 0.49


,metric,estimate,CI_low,CI_high
0,accuracy,0.8197,0.7213,0.9016
1,precision_ASD,0.9574,0.9111,1.0000
2,recall_ASD,0.8333,0.7222,0.9259
3,f1_ASD,0.8911,0.8247,0.9445
4,macro_f1,0.6836,0.5564,0.8206
5,recall_non_ASD,0.7143,0.4286,1.0000
6,f1_non_ASD,0.4762,0.2667,0.7000
7,roc_auc,0.8095,0.6111,0.9551


,predicted_non_ASD,predicted_ASD
actual_non_ASD,5,2
actual_ASD,9,45


Seed 43 - Tweet Metrics


... (18 outputs left out)

,predicted_non_ASD,predicted_ASD
actual_non_ASD,1,6
actual_ASD,4,50


,seed,threshold,accuracy,precision_ASD,recall_ASD,f1_ASD,macro_f1,recall_non_ASD,f1_non_ASD,roc_auc
0,42,0.49,0.8197,0.9574,0.8333,0.8911,0.6836,0.7143,0.4762,0.8095
1,43,0.46,0.7869,0.9020,0.8519,0.8762,0.5557,0.2857,0.2353,0.5503
2,44,0.46,0.7869,0.9362,0.8148,0.8713,0.6261,0.5714,0.3810,0.7672
3,45,0.50,0.8197,1.0000,0.7963,0.8866,0.7233,1.0000,0.5600,0.9788
4,46,0.45,0.8361,0.8929,0.9259,0.9091,0.5379,0.1429,0.1667,0.7698


,metric,mean,std
0,accuracy,0.8098,0.0220
1,precision_ASD,0.9377,0.0435
2,recall_ASD,0.8444,0.0500
3,f1_ASD,0.8869,0.0147
4,macro_f1,0.6253,0.0798
5,recall_non_ASD,0.5429,0.3411
6,f1_non_ASD,0.3638,0.1634
7,roc_auc,0.7751,0.1528


,seed,basis,TextCNN,CI
0,42,meta checkpoint + probabilities,saved probabilities/predictions,saved
1,43,meta checkpoint + probabilities,saved probabilities/predictions,saved
2,44,meta checkpoint + probabilities,saved probabilities/predictions,saved
3,45,meta checkpoint + probabilities,saved probabilities/predictions,saved
4,46,meta checkpoint + probabilities,saved probabilities/predictions,saved
